# Sandbox extendido del pipeline sin malla -- eje + plano, 100% auto-contenido

Version extendida de `sandbox_pipeline_sin_malla.ipynb`: ahora corre **eje
(axis_v06) y plano (plane_v04_1) en paralelo**, cada uno sobre sus propios 30
objetos reales (10 BUENO / 10 MEDIO / 10 MALO), seleccionados con
`Mapping/select_sandbox_objects.py` a partir de `axis_v06_nomesh`
(por `angular_error_deg`) y `plane_v04_1_nomesh` (por F1 por-objeto
aproximado -- ver el docstring de ese script para por que `f1_ref` en si no
es una metrica por objeto).

```
1. Listas de objetos (reales, ya seleccionadas)
2. Copiar .obj/.txt y renders existentes -> carpeta propia en Experiments/
3. Prompts (axis_v06 y plane_v04_1, editables en celdas)
4. Geometria: camara, triangulacion (eje Y plano), filtro fondo/objeto
5. Molmo2: carga del modelo + inferencia (inline)          [necesita GPU]
6. Correr inferencia (eje y plano, por separado)
7. Estimacion sin malla (eje: triangulation; plano: triangulation_multiplane)
8. Evaluacion contra GT (eje y plano)
9. Resultados por objeto, con su categoria BUENO/MEDIO/MALO
10. Visualizacion 2D/3D (opcional)
```

**Ningun paso llama a un script .py del repo** -- todo el codigo esta
copiado inline (prompts, geometria, parseo de Molmo2, metricas) para poder
editarlo directo en las celdas. Es una copia funcional de
`MolmoPointing/molmo_multiview_runner.py`, `pipeline_common/camera.py`,
`pipeline_common/triangulation.py`, `Mapping/estimate_symmetry_no_mesh.py` y
`Mapping/evaluate.py` -- si una variante gana aca, hay que trasplantarla a
mano a esos archivos para que entre al sweep completo de 850 objetos.

**Incluye `SDE_ref`/`F1_ref`** (formulas exactas de `Mapping/evaluate.py`,
requieren `gpytoolbox` -- disponible en el servidor). `SDE_ref` aplica a eje
Y plano; `F1_ref` solo a plano (no existe una convencion de F1 para eje, ni
en el repo de referencia ni en la literatura). Se reportan tanto para el
conjunto completo como para cada subconjunto BUENO/MEDIO/MALO por separado.

Los `.obj`/`.txt` estan en `data/objects/curated_{axis,plane}_sym_obj/` y
los renders en `data/renders/{axis,plane}_sym/<object_id>/224/flat/` --
misma ruta `data/` que usan todos los scripts `.py` del repo (esto corre en
el servidor).

## 0. Setup

In [1]:
import json
import re
import shutil
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import trimesh
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor

REPO_ROOT = Path(r"/home/pespinoz/Symmetry-Detection-Using-Multimodal-Vision-Language-Models")
DATA_ROOT = Path(r"/home/pespinoz/data")  # solo lectura -- fuente de .obj/.txt/renders
# En el servidor, ajustar a algo como Path("../data") si corres desde la raiz del repo con rutas relativas.


SANDBOX_ROOT    = REPO_ROOT / "Experiments" 
SANDBOX_OBJECTS = SANDBOX_ROOT / "objects"
SANDBOX_RENDERS = SANDBOX_ROOT / "renders"

OBJECTS_SUBDIR_MAP = {"axis_sym": "curated_axis_sym_obj", "plane_sym": "curated_plane_sym_obj"}
SIZE, LIGHTING = 224, "flat"
DEFAULT_FOV    = 60.0
VIEW_GROUPS    = [6, 14, 26]

MODEL_ID = "allenai/Molmo2-8B"

EDGE_ON_THRESH_DEFAULT   = 0.5    # plano: |cos(angulo)| por debajo = vista "de canto"
DUP_ANGLE_THRESH_DEFAULT = 15.0   # plano: grados; planos candidatos mas cerca que esto = duplicados

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)
plt.rcParams.update({"figure.dpi": 110})

## 1. Listas de objetos (reales, 10 BUENO / 10 MEDIO / 10 MALO cada uno)

Generadas con `python Mapping/select_sandbox_objects.py ...` sobre
`axis_v06_nomesh` (angular_error) y `plane_v04_1_nomesh` (F1 por-objeto
aproximado). Formato de entrada `{CATEGORIA}_{object_id}_2d`.

In [2]:
def parse_entry(entry: str) -> tuple[str, str]:
    """'BUENO_89cb9b2ad175b833cadf6344ec272e8_2d' -> ('BUENO', '89cb9b2ad175b833cadf6344ec272e8')."""
    parts = entry.split("_")
    categoria = parts[0]
    object_id = next(p for p in parts[1:] if len(p) >= 30 and all(c in "0123456789abcdef" for c in p))
    return categoria, object_id


RAW_OBJECT_LIST_AXIS = [
    "BUENO_941271c5d9b192eaccd8f9b9403fd602_2d",
    "BUENO_e0725fd7859fa238ff67c12005f72d2_2d",
    "BUENO_89cb9b2ad175b833cadf6344ec272e8_2d",
    "BUENO_dc2c07ff617d1da0197a35146ee825cd_2d",
    "BUENO_b35973652d9a526c64848cfde3847b25_2d",
    "BUENO_28d39c9673775f601a3b47d6d0918049_2d",
    "BUENO_3a29634236aa0596f9e8cd846ef13776_2d",
    "BUENO_9fe7e6a7bf8ca964efad53eb3f0b36fa_2d",
    "BUENO_5e914dd726313181b1b4b514128408b0_2d",
    "BUENO_b2acbb6717c7a842fcb8d8c6d4df8143_2d",
    "MEDIO_8d457deaf22394da65c5c31ac688ec4_2d",
    "MEDIO_d3a3d52234e722825208aab875b932bc_2d",
    "MEDIO_94fdbb748526dae4ea2d70ab68cd1d2_2d",
    "MEDIO_297d929269bb62da43fdcbcacbbed64c_2d",
    "MEDIO_6b8b2cb01c376064c8724d5673a063a6_2d",
    "MEDIO_f0611ec9ff89209bf10c4513652c1c5e_2d",
    "MEDIO_33b77c66e1f849b790c4e2a44fddf755_2d",
    "MEDIO_1b4d7803a3298f8477bdcb8816a3fac9_2d",
    "MEDIO_d851cbc873de1c4d3b6eb309177a6753_2d",
    "MEDIO_618d55a791b8280cf256a8c3e3396495_2d",
    "MALO_64f61c9c81e3eb7b8aaae3d020f5ddf8_2d",
    "MALO_66cec0a2ab63d9101b6c273f8ff0e8b6_2d",
    "MALO_8adc0ce79962ac2021072d05c97a5e0a_2d",
    "MALO_6e1fe96adbb5ffba8bae2d07dadd1b5d_2d",
    "MALO_40aee078f0390b7134076b5960251711_2d",
    "MALO_56efabe64ce30e08d7bc2c778b186f58_2d",
    "MALO_5dd2f4d3b253058dd554ab0a45f30de7_2d",
    "MALO_6267ef99cbfaea7741cf86c757faf4f9_2d",
    "MALO_955143d7f0b5c70fef76898f881b76a_2d",
    "MALO_e656d6586d481f41eb69804478f9c547_2d",
]

RAW_OBJECT_LIST_PLANE = [
    "BUENO_109a19840a8fbc774c3aee8e9d3a6ffa_2d",
    "BUENO_1541e36e8dc2d84caed2201239784a35_2d",
    "BUENO_15e2d1fe9a4b28e68fb8c03013603b0c_2d",
    "BUENO_1655608678a4f9ffbc7bb5239e53ea6f_2d",
    "BUENO_1ad4c572e0fd6a576e1e9a13188ab4bb_2d",
    "BUENO_1b6a5fc808388138cb2a965e75be701c_2d",
    "BUENO_2620701a50216dbed0b36851d61b6fca_2d",
    "BUENO_2912f56d135e2f97b20cb946bceb58f_2d",
    "BUENO_2b25e49c58ae0e292056b4bd5d870b47_2d",
    "BUENO_3279edef6a631940ea41b93204b74265_2d",
    "MEDIO_82d8391c62e197161282d4d9178caa92_2d",
    "MEDIO_82e0e96bd89956d9f330eedffaf3339e_2d",
    "MEDIO_835e01af50296235aefda81565fede7_2d",
    "MEDIO_8399987b9bca0dd0e63a9e8397b31118_2d",
    "MEDIO_83c90f7b104816ecc748af2814b558c4_2d",
    "MEDIO_8497e02fa1662113776d8bc79b9caa2c_2d",
    "MEDIO_84aa911799cd87b4ad5067eac75a07f7_2d",
    "MEDIO_84efcf2796fad0d2917fe9209d09e56e_2d",
    "MEDIO_854ffe2c60091876e07a1c4b84dfd325_2d",
    "MEDIO_8596664f3d7925cdfdeb515ad63cf4b0_2d",
    "MALO_a4b457c41180e0b9fe52ffd0e748a1ab_2d",
    "MALO_a85b9a3ace923e424721d5612f98ae26_2d",
    "MALO_ac52cf0b598e930ab38d3c03866c1379_2d",
    "MALO_ad86354fb5faf1c98a4820926b2a786_2d",
    "MALO_d0c3bf270f9e04eb362845c6edb57fc_2d",
    "MALO_d2553e5fc4f1527cfeae521e94848af6_2d",
    "MALO_d2a511578a387365ede9471c962dd6ba_2d",
    "MALO_dacf794f991fc0173e9b7c3ab7636200_2d",
    "MALO_dd2a4c416625f29c4f57a7ededfb3bde_2d",
    "MALO_de519752147a225032387cdb9b2a84d5_2d",
]

OBJECT_LIST_AXIS  = [parse_entry(e) for e in RAW_OBJECT_LIST_AXIS]
OBJECT_LIST_PLANE = [parse_entry(e) for e in RAW_OBJECT_LIST_PLANE]
CATEGORY_BY_ID_AXIS  = {oid: cat for cat, oid in OBJECT_LIST_AXIS}
CATEGORY_BY_ID_PLANE = {oid: cat for cat, oid in OBJECT_LIST_PLANE}

print(f"axis_sym : {len(OBJECT_LIST_AXIS)} objetos")
print(f"plane_sym: {len(OBJECT_LIST_PLANE)} objetos")

axis_sym : 30 objetos
plane_sym: 30 objetos


## 2. Copiar datos al sandbox

Solo copia lo minimo: `.obj`/`.txt` (GT) y, de cada render ya generado, el
`manifest.json` + `metadata_all.json` (poses de camara) + los PNG -- **sin
volver a renderizar**. No copia ningun `molmo_multiview*`/`predicted_*`/
`eval_*` -- eso se genera de cero en el sandbox.

In [3]:
def copy_object_data(symmetry_type: str, object_list: list[tuple[str, str]]) -> None:
    objects_subdir = OBJECTS_SUBDIR_MAP[symmetry_type]
    sandbox_objects_dir = SANDBOX_OBJECTS / objects_subdir
    sandbox_objects_dir.mkdir(parents=True, exist_ok=True)

    for cat, oid in object_list:
        src_objects_dir = DATA_ROOT / "objects" / objects_subdir
        for ext in (".obj", ".txt"):
            src, dst = src_objects_dir / f"{oid}{ext}", sandbox_objects_dir / f"{oid}{ext}"
            if not dst.exists():
                assert src.exists(), f"No encontrado: {src}"
                shutil.copy2(src, dst)

        src_render_dir = DATA_ROOT / "renders" / symmetry_type / oid / str(SIZE) / LIGHTING
        dst_render_dir = SANDBOX_RENDERS / symmetry_type / oid / str(SIZE) / LIGHTING
        dst_render_dir.mkdir(parents=True, exist_ok=True)
        assert src_render_dir.exists(), f"No encontrado: {src_render_dir}"

        for meta_file in ("manifest.json", "metadata_all.json"):
            src_meta, dst_meta = src_render_dir / meta_file, dst_render_dir / meta_file
            if src_meta.exists() and not dst_meta.exists():
                shutil.copy2(src_meta, dst_meta)

        n_copied = 0
        for png in src_render_dir.glob("*.png"):
            dst_png = dst_render_dir / png.name
            if not dst_png.exists():
                shutil.copy2(png, dst_png)
                n_copied += 1
        print(f"[{symmetry_type:9s}][{cat:6s}] {oid}: {n_copied} PNG copiados")


copy_object_data("axis_sym", OBJECT_LIST_AXIS)
copy_object_data("plane_sym", OBJECT_LIST_PLANE)
print(f"\nSandbox listo en: {SANDBOX_ROOT}")

[axis_sym ][BUENO ] 941271c5d9b192eaccd8f9b9403fd602: 0 PNG copiados
[axis_sym ][BUENO ] e0725fd7859fa238ff67c12005f72d2: 0 PNG copiados
[axis_sym ][BUENO ] 89cb9b2ad175b833cadf6344ec272e8: 0 PNG copiados
[axis_sym ][BUENO ] dc2c07ff617d1da0197a35146ee825cd: 0 PNG copiados
[axis_sym ][BUENO ] b35973652d9a526c64848cfde3847b25: 0 PNG copiados
[axis_sym ][BUENO ] 28d39c9673775f601a3b47d6d0918049: 0 PNG copiados
[axis_sym ][BUENO ] 3a29634236aa0596f9e8cd846ef13776: 0 PNG copiados
[axis_sym ][BUENO ] 9fe7e6a7bf8ca964efad53eb3f0b36fa: 0 PNG copiados
[axis_sym ][BUENO ] 5e914dd726313181b1b4b514128408b0: 0 PNG copiados
[axis_sym ][BUENO ] b2acbb6717c7a842fcb8d8c6d4df8143: 0 PNG copiados
[axis_sym ][MEDIO ] 8d457deaf22394da65c5c31ac688ec4: 0 PNG copiados
[axis_sym ][MEDIO ] d3a3d52234e722825208aab875b932bc: 0 PNG copiados
[axis_sym ][MEDIO ] 94fdbb748526dae4ea2d70ab68cd1d2: 0 PNG copiados
[axis_sym ][MEDIO ] 297d929269bb62da43fdcbcacbbed64c: 0 PNG copiados
[axis_sym ][MEDIO ] 6b8b2cb01c376064c8

[axis_sym ][MALO  ] e656d6586d481f41eb69804478f9c547: 0 PNG copiados
[plane_sym][BUENO ] 109a19840a8fbc774c3aee8e9d3a6ffa: 0 PNG copiados
[plane_sym][BUENO ] 1541e36e8dc2d84caed2201239784a35: 0 PNG copiados
[plane_sym][BUENO ] 15e2d1fe9a4b28e68fb8c03013603b0c: 0 PNG copiados
[plane_sym][BUENO ] 1655608678a4f9ffbc7bb5239e53ea6f: 0 PNG copiados
[plane_sym][BUENO ] 1ad4c572e0fd6a576e1e9a13188ab4bb: 0 PNG copiados
[plane_sym][BUENO ] 1b6a5fc808388138cb2a965e75be701c: 0 PNG copiados
[plane_sym][BUENO ] 2620701a50216dbed0b36851d61b6fca: 0 PNG copiados
[plane_sym][BUENO ] 2912f56d135e2f97b20cb946bceb58f: 0 PNG copiados
[plane_sym][BUENO ] 2b25e49c58ae0e292056b4bd5d870b47: 0 PNG copiados
[plane_sym][BUENO ] 3279edef6a631940ea41b93204b74265: 0 PNG copiados
[plane_sym][MEDIO ] 82d8391c62e197161282d4d9178caa92: 0 PNG copiados
[plane_sym][MEDIO ] 82e0e96bd89956d9f330eedffaf3339e: 0 PNG copiados
[plane_sym][MEDIO ] 835e01af50296235aefda81565fede7: 0 PNG copiados
[plane_sym][MEDIO ] 8399987b9bca0dd0

## 3. Prompts (editables aca, sin tocar archivos .txt)

`axis_v06` (mejor eje) y `plane_v04_1` (mejor plano por F1) -- copia
funcional de `MolmoPointing/prompts/{axis,plane}/...`.

In [4]:
PROMPT_ID_AXIS = "axis_v06"

PROMPT_SINGLE_AXIS = """You are given ONE image of a 3D object.

The object has ONE dominant rotational symmetry axis.

Your task is to identify the two poles of the rotation axis: the topmost and bottommost points where the axis exits the object's surface.

Return:
- obj_id 1: the TOP pole -- topmost visible point on the rotation axis (at the horizontal center of the topmost surface)
- obj_id 2: the BOTTOM pole -- bottommost visible point on the rotation axis (at the horizontal center of the bottommost surface)

IMPORTANT RULES:
- Both points MUST lie ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- obj_id 1 MUST be above obj_id 2 (smaller Y value).
- The two points MUST be as far apart vertically as possible.
- Both points MUST lie on the visible object surface.
- For flat-topped or flat-bottomed objects, place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point on the visible surface that is equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Verify: draw an imaginary horizontal line through each point -- the point should be equidistant from the left and right edges at that height, regardless of whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2">"""


PROMPT_MULTI_AXIS = """You are given multiple views of the SAME 3D object.

The object has ONE dominant axis of rotational symmetry.

For each image, identify the TOP and BOTTOM poles of the global rotation axis -- the points where the axis exits the object's surface at the very top and very bottom.

For each image:
1. Locate where the global axis exits at the top and bottom of the object.
2. Return:
   - obj_id 1: the TOP pole (topmost point on the axis, at the horizontal center of the topmost surface),
   - obj_id 2: the BOTTOM pole (bottommost point on the axis, at the horizontal center of the bottommost surface).

IMPORTANT RULES:
- Both points MUST be ON the rotation axis -- at the horizontal center of the object at that height, NOT on the lateral silhouette.
- Use the SAME global axis consistently across all views.
- obj_id 1 MUST be above obj_id 2 in each image.
- For flat surfaces (e.g., flat-topped objects), place the pole at the geometric center of the top or bottom face.
- For ROUNDED or CURVED tops/bottoms (no single flat face -- e.g. a dome, a sphere cap, a rounded knob), place the pole at the center of curvature of that rounded region: the point equidistant from the left and right silhouette edges at that region's topmost/bottommost extent.
- Infer the global axis from ALL views jointly before answering.
- Verify consistency: the TOP pole should correspond to the same geometric point on the object across all views, whether the surface there is flat or curved.
- Do NOT place points on the lateral silhouette edges.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 Xtop Ytop 2 Xbottom Ybottom; 2 1 Xtop Ytop 2 Xbottom Ybottom; 3 1 Xtop Ytop 2 Xbottom Ybottom">

Where each entry is: image_index obj_id X Y
- obj_id 1 = TOP pole of the rotation axis
- obj_id 2 = BOTTOM pole of the rotation axis

Return ONLY the <points ...> block."""

print(f"Prompt eje cargado: {PROMPT_ID_AXIS}")

Prompt eje cargado: axis_v06


In [5]:
PROMPT_ID_PLANE = "plane_v04_1"

PROMPT_SINGLE_PLANE = """You are given ONE image of a 3D object.

The object has ONE dominant plane of reflective symmetry.

Your task is to identify the horizontal midpoints of the object's left-right extent at two different heights: one near the TOP and one near the BOTTOM of the object.

These midpoints lie on the symmetry plane's projected trace.

Return:
- obj_id 1: the horizontal midpoint of the left-right extent near the TOP of the object
- obj_id 2: the horizontal midpoint of the left-right extent near the BOTTOM of the object

The horizontal midpoint at height Y = (X_left_edge + X_right_edge) / 2 at that height.

IMPORTANT RULES:
- Both points MUST lie on the symmetry plane trace (horizontal center of the object at that height).
- obj_id 1 MUST be in the upper half of the object's visible extent.
- obj_id 2 MUST be in the lower half of the object's visible extent.
- Both points MUST lie on or within the visible object.
- Do NOT place points on the silhouette edges -- place them at the HORIZONTAL CENTER between the edges.
- Do NOT collapse both points to the same height.
- Use the object's bilateral geometry to verify that the chosen midpoints are consistent with the symmetry plane direction.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2">"""


PROMPT_MULTI_PLANE = """You are given multiple views of the SAME 3D object.

The object has ONE dominant plane of reflective symmetry.

Step 1: Identify the global symmetry plane from ALL views together -- it divides the object into two mirror halves.
Step 2: For each image, find the horizontal midpoints of the object's left-right extent at two heights (top and bottom) that lie on the plane's projected trace.

For each image:
1. Find the horizontal midpoint at the top of the object and at the bottom.
2. Return:
   - obj_id 1: horizontal midpoint near the TOP (upper half),
   - obj_id 2: horizontal midpoint near the BOTTOM (lower half).

The horizontal midpoint at height Y = (X_left_edge + X_right_edge) / 2 at that height.

IMPORTANT RULES:
- Use the SAME global symmetry plane consistently across all views.
- Both points in each image must be on the plane's projected trace (horizontal center of the object at that height).
- obj_id 1 MUST be above obj_id 2 in every image.
- Both points MUST lie within the visible object bounds.
- Do NOT place points on the silhouette edges -- only on the horizontal center between them.
- Infer the global plane from ALL views jointly before answering.
- Verify cross-view consistency: the midpoint direction across images should reflect the same global plane orientation.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 Xtop Ytop 2 Xbottom Ybottom; 2 1 Xtop Ytop 2 Xbottom Ybottom; 3 1 Xtop Ytop 2 Xbottom Ybottom">

Where each entry is: image_index obj_id X Y
- obj_id 1 = horizontal midpoint near the TOP of the object
- obj_id 2 = horizontal midpoint near the BOTTOM of the object

Return ONLY the <points ...> block."""

print(f"Prompt plano cargado: {PROMPT_ID_PLANE}")

Prompt plano cargado: plane_v04_1


In [44]:
PROMPT_ID_AXIS = "axis_v07_6pts"

PROMPT_SINGLE_AXIS = """You are given ONE image of a 3D object.

The object has ONE dominant rotational symmetry axis.

Your task is to place SIX points along the projected axis line — the vertical centerline of the object as seen in this image. These six points together define a straight line in the image that represents the axis projection.

Return six points ordered from top to bottom:
- obj_id 1: TOP pole — topmost point on the axis (horizontal center of the topmost surface)
- obj_id 2: axis centerline at ~80% of the object's vertical extent (upper region)
- obj_id 3: axis centerline at ~60% of the object's vertical extent (upper-middle)
- obj_id 4: axis centerline at ~40% of the object's vertical extent (lower-middle)
- obj_id 5: axis centerline at ~20% of the object's vertical extent (lower region)
- obj_id 6: BOTTOM pole — bottommost point on the axis (horizontal center of the bottommost surface)

IMPORTANT RULES:
- ALL six points MUST lie on the rotation axis — at the HORIZONTAL CENTER of the object at each height, NOT on the lateral silhouette.
- ALL six points MUST be inside the visible object (on its surface), never on the background.
- The six points MUST be at six DIFFERENT heights, strictly ordered top (obj_id 1) to bottom (obj_id 6).
- The six points MUST be approximately collinear in the image — they should all lie on the same straight projected line.
- Maximize the vertical spread: obj_id 1 and obj_id 6 must be as far apart as possible.
- At each height, verify: draw an imaginary horizontal line — the point must be equidistant from the left and right silhouette edges at that height.
- For flat tops/bottoms: place the pole at the geometric center of the face.
- For rounded tops/bottoms: place the pole at the point equidistant from left and right silhouette edges at the topmost/bottommost extent.
- Do NOT place points on the lateral silhouette edges.
- SELF-CHECK: if you drew a straight line through your six points, it should follow the vertical axis of the object from top to bottom.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6">"""


PROMPT_MULTI_AXIS = """You are given multiple views of the SAME 3D object.

The object has ONE dominant rotational symmetry axis.

For each image, place SIX points along the projected axis line — the centerline of the object as seen from that viewpoint. These six points together define a straight line in each image representing the axis projection.

For each image:
1. Infer the global rotation axis from ALL views jointly.
2. Project that axis into the current view and identify the centerline.
3. Place six points along that centerline, from top to bottom:
   - obj_id 1: TOP pole (topmost axis point, horizontal center of topmost surface)
   - obj_id 2: axis centerline at ~80% of object's vertical extent
   - obj_id 3: axis centerline at ~60% of object's vertical extent
   - obj_id 4: axis centerline at ~40% of object's vertical extent
   - obj_id 5: axis centerline at ~20% of object's vertical extent
   - obj_id 6: BOTTOM pole (bottommost axis point, horizontal center of bottommost surface)

IMPORTANT RULES:
- ALL six points MUST be ON the rotation axis — at the horizontal center of the object at each height, NOT on the lateral silhouette.
- ALL six points MUST be inside the visible object, never on the background.
- Use the SAME global axis consistently across all views.
- The six points MUST be at six DIFFERENT heights, strictly ordered top (obj_id 1) to bottom (obj_id 6) in EVERY image.
- The six points MUST be approximately collinear in each image — they should lie on the same projected line.
- Maximize vertical spread: obj_id 1 and obj_id 6 must be as far apart as possible in each image.
- At each height, the point must be equidistant from the left and right silhouette edges.
- Do NOT place points on the lateral silhouette edges.
- Infer the global axis from ALL views jointly before answering.
- SELF-CHECK per image: your six points should form a straight line in the image following the object's vertical centerline.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6; 2 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6; 3 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6">

Where each entry is: image_index obj_id X Y (repeated for obj_id 1 through 6)

Return ONLY the <points ...> block."""

print(f"Prompt eje cargado: {PROMPT_ID_AXIS}")

Prompt eje cargado: axis_v07_6pts


In [45]:
PROMPT_ID_PLANE = "plane_v05_6pts"

PROMPT_SINGLE_PLANE = """You are given ONE image of a 3D object.

The object has ONE dominant plane of reflective symmetry.

Your task is to place SIX points along the plane's projected trace — the line where the symmetry plane intersects the visible object, dividing its two mirror halves. These six points together define a straight line in the image representing the plane trace.

Return six points ordered from top to bottom:
- obj_id 1: trace point near the TOP of the object (upper boundary of the visible trace)
- obj_id 2: trace point at ~80% of the object's vertical extent
- obj_id 3: trace point at ~60% of the object's vertical extent (upper-middle)
- obj_id 4: trace point at ~40% of the object's vertical extent (lower-middle)
- obj_id 5: trace point at ~20% of the object's vertical extent
- obj_id 6: trace point near the BOTTOM of the object (lower boundary of the visible trace)

Each trace point is the horizontal midpoint between the left and right edges of the object at that height:
midpoint_X = (X_left_edge + X_right_edge) / 2

IMPORTANT RULES:
- ALL six points MUST lie on the symmetry plane trace (horizontal center between left and right edges at each height).
- ALL six points MUST be inside the visible object, never on the background.
- The six points MUST be at six DIFFERENT heights, strictly ordered top (obj_id 1) to bottom (obj_id 6).
- The six points MUST be approximately collinear in the image — they should all lie on the same straight projected trace line.
- Maximize vertical spread: obj_id 1 and obj_id 6 must be as far apart as possible.
- Do NOT place points on the silhouette edges — only on the horizontal center between them.
- SELF-CHECK: if you drew a straight line through your six points, it should follow the plane's projected trace from top to bottom of the object.

Output ONLY:

<points coords="1 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6">"""


PROMPT_MULTI_PLANE = """You are given multiple views of the SAME 3D object.

The object has ONE dominant plane of reflective symmetry.

For each image, place SIX points along the plane's projected trace — the line dividing the object into two mirror halves as seen from that viewpoint. These six points together define a straight line in each image.

Step 1: Identify the global symmetry plane from ALL views jointly.
Step 2: For each image, project that plane's trace and place six points along it.

For each image:
1. Find the plane's projected trace in that view (the line of bilateral symmetry as seen from that camera).
2. Place six points along the trace, from top to bottom:
   - obj_id 1: trace point near the TOP (upper boundary)
   - obj_id 2: trace point at ~80% of the object's vertical extent
   - obj_id 3: trace point at ~60% of the object's vertical extent
   - obj_id 4: trace point at ~40% of the object's vertical extent
   - obj_id 5: trace point at ~20% of the object's vertical extent
   - obj_id 6: trace point near the BOTTOM (lower boundary)

Each trace point is: midpoint_X = (X_left_edge + X_right_edge) / 2 at that height, where left/right is measured relative to the symmetry plane's orientation in that view.

IMPORTANT RULES:
- ALL six points MUST lie on the plane's projected trace (horizontal center of the bilateral extent at each height).
- ALL six points MUST be inside the visible object, never on the background.
- Use the SAME global symmetry plane consistently across all views.
- The six points MUST be at six DIFFERENT heights, strictly ordered top (obj_id 1) to bottom (obj_id 6) in EVERY image.
- The six points MUST be approximately collinear in each image — they should define a straight projected trace line.
- Maximize vertical spread: obj_id 1 and obj_id 6 must be as far apart as possible in each image.
- Do NOT place points on the silhouette edges — only at the bilateral center.
- Infer the global plane from ALL views jointly before answering.
- SELF-CHECK per image: your six points should form a straight line in the image following the plane trace from top to bottom of the object.

Output format (one entry per image, separated by semicolons):

<points coords="1 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6; 2 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6; 3 1 X1 Y1 2 X2 Y2 3 X3 Y3 4 X4 Y4 5 X5 Y5 6 X6 Y6">

Where each entry is: image_index obj_id X Y (repeated for obj_id 1 through 6)

Return ONLY the <points ...> block."""

print(f"Prompt plano cargado: {PROMPT_ID_PLANE}")

Prompt plano cargado: plane_v05_6pts


## 4. Geometria: camara, triangulacion (eje Y plano), filtro fondo/objeto

Copia funcional de `pipeline_common/camera.py` + `pipeline_common/triangulation.py`
+ la parte de plano de `Mapping/estimate_symmetry_no_mesh.py`.

In [6]:
# --- pipeline_common/camera.py ---

def molmo_to_ndc(x: float, y: float) -> tuple[float, float]:
    ndc_x = (x / 1000.0) * 2.0 - 1.0
    ndc_y = 1.0 - (y / 1000.0) * 2.0
    return ndc_x, ndc_y


def build_camera_rays(ndc_x: float, ndc_y: float, R: list, T: list,
                      fov_deg: float, image_size: int) -> tuple[np.ndarray, np.ndarray]:
    """PyTorch3D row-vector convention: p_cam = p_world @ R + T, camera center = -(R @ T)."""
    R_np, T_np = np.array(R, dtype=np.float64), np.array(T, dtype=np.float64)
    ray_origin = -(R_np @ T_np)
    half_tan = np.tan(np.deg2rad(fov_deg) / 2.0)
    dir_cam = np.array([ndc_x * half_tan, ndc_y * half_tan, 1.0], dtype=np.float64)
    dir_world = R_np @ dir_cam
    dir_world /= np.linalg.norm(dir_world)
    return ray_origin, dir_world


# --- pipeline_common/triangulation.py ---

def ray_dir_for_point(x: float, y: float, R: list, T: list, fov_deg: float, image_size: int):
    ndc_x, ndc_y = molmo_to_ndc(x, y)
    return build_camera_rays(ndc_x, ndc_y, R, T, fov_deg, image_size)


def view_forward_direction(R: list, T: list, fov_deg: float, image_size: int) -> np.ndarray:
    _, direction = build_camera_rays(0.0, 0.0, R, T, fov_deg, image_size)
    return direction


def interpretation_plane_normal(dir_a: np.ndarray, dir_b: np.ndarray):
    """Normal of the plane containing the camera center and two rays from it
    (Bartoli & Sturm 2005). None if the two rays are (numerically) parallel."""
    n = np.cross(dir_a, dir_b)
    norm = np.linalg.norm(n)
    if norm < 1e-9:
        return None
    return n / norm



def triangulate_line(camera_centers: list, plane_normals: list) -> tuple[np.ndarray, np.ndarray]:
    N = np.asarray(plane_normals, dtype=np.float64)
    C = np.asarray(camera_centers, dtype=np.float64)
    _, _, Vt = np.linalg.svd(N)
    direction = Vt[-1]
    direction /= np.linalg.norm(direction)
    b = np.einsum("ij,ij->i", N, C)
    point, *_ = np.linalg.lstsq(N, b, rcond=None)
    return point, direction



def widest_pair(pts: list):
    """De TODOS los puntos que trae una vista, el par con mayor distancia
    euclidiana en pixeles -- None si hay menos de 2 puntos. Con exactamente 2
    puntos, es identico a tomar esos dos."""
    if len(pts) < 2:
        return None
    best_pair, best_dist_sq = None, -1.0
    for i in range(len(pts)):
        for j in range(i + 1, len(pts)):
            dx = pts[i]["x"] - pts[j]["x"]
            dy = pts[i]["y"] - pts[j]["y"]
            dist_sq = dx * dx + dy * dy
            if dist_sq > best_dist_sq:
                best_dist_sq, best_pair = dist_sq, (pts[i], pts[j])
    return best_pair


def get_point_by_obj_id(pts: list, obj_id: int):
    return next((p for p in pts if p["obj_id"] == obj_id), None)

In [7]:
# --- filtro fondo/objeto (Mapping/estimate_symmetry_no_mesh.py::filter_points_on_object) ---

BACKGROUND_THRESH = 250  # RGB > esto en los 3 canales = fondo blanco (PyTorch3D HardFlatShader default)


def is_on_object(pixel: np.ndarray) -> bool:
    return not bool(np.all(pixel[:3] > BACKGROUND_THRESH))


def molmo_xy_to_pixel(x: float, y: float, img_w: int, img_h: int) -> tuple[int, int]:
    px = int(round((x / 1000.0) * img_w))
    py = int(round((y / 1000.0) * img_h))
    return min(max(px, 0), img_w - 1), min(max(py, 0), img_h - 1)


def filter_points_on_object(points_by_image: dict, images_sent: list, render_dir: Path,
                            min_points: int = 2, image_cache: dict | None = None) -> dict:
    if image_cache is None:
        image_cache = {}
    filtered = {}
    for img_idx_str, pts in points_by_image.items():
        if not pts:
            filtered[img_idx_str] = pts
            continue
        cam = images_sent[int(img_idx_str)]
        filename = cam["filename"]
        if filename not in image_cache:
            img_path = render_dir / filename
            image_cache[filename] = np.array(Image.open(img_path).convert("RGB")) if img_path.exists() else None
        img = image_cache[filename]
        if img is None:
            filtered[img_idx_str] = pts
            continue
        img_h, img_w = img.shape[0], img.shape[1]
        kept = []
        for p in pts:
            px, py = molmo_xy_to_pixel(p["x"], p["y"], img_w, img_h)
            if is_on_object(img[py, px]):
                kept.append(p)
        filtered[img_idx_str] = kept if len(kept) >= min_points else []
    return filtered


# --- eje: Mapping/estimate_symmetry_no_mesh.py::estimate_axis_no_mesh ---

def estimate_axis_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int):
    centers, normals = [], []
    for img_idx_str, pts in points_by_image.items():
        pair = widest_pair(pts)
        if pair is None:
            continue
        p_a, p_b = pair
        cam = images_sent[int(img_idx_str)]
        C, d_a = ray_dir_for_point(p_a["x"], p_a["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d_b = ray_dir_for_point(p_b["x"], p_b["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d_a, d_b)
        if n is None:
            continue
        centers.append(C)
        normals.append(n)
    if len(normals) < 2:
        raise ValueError(f"need >=2 valid views, got {len(normals)}")
    point, direction = triangulate_line(centers, normals)
    return {"direction": direction.tolist(), "origin": point.tolist(), "n_views_used": len(normals)}

In [8]:
# --- plano: Mapping/estimate_symmetry_no_mesh.py::_line_from_view_pair / estimate_plane_no_mesh / detect_planes_no_mesh ---

def _line_from_view_pair(view_i: int, view_j: int, points_by_image: dict, images_sent: list,
                         fov_deg: float, image_size: int):
    centers, normals = [], []
    for idx in (view_i, view_j):
        pts = points_by_image.get(str(idx), [])
        p1, p2 = get_point_by_obj_id(pts, 1), get_point_by_obj_id(pts, 2)
        if p1 is None or p2 is None:
            return None
        cam = images_sent[idx]
        C, d1 = ray_dir_for_point(p1["x"], p1["y"], cam["R"], cam["T"], fov_deg, image_size)
        _, d2 = ray_dir_for_point(p2["x"], p2["y"], cam["R"], cam["T"], fov_deg, image_size)
        n = interpretation_plane_normal(d1, d2)
        if n is None:
            return None
        centers.append(C)
        normals.append(n)
    return triangulate_line(centers, normals)


def estimate_plane_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int,
                           edge_on_thresh: float = EDGE_ON_THRESH_DEFAULT) -> dict:
    """Esquema de 4 pasos: lineas candidatas por par de vistas -> normales
    candidatas por producto cruzado -> puntuar 'de canto' -> refit SVD."""
    view_idxs = sorted(
        int(k) for k, pts in points_by_image.items()
        if get_point_by_obj_id(pts, 1) is not None and get_point_by_obj_id(pts, 2) is not None
    )
    if len(view_idxs) < 4:
        raise ValueError(f"need >=4 valid views (2 independent pairs), got {len(view_idxs)}")

    pair_lines = []
    for i, j in combinations(view_idxs, 2):
        res = _line_from_view_pair(i, j, points_by_image, images_sent, fov_deg, image_size)
        if res is not None:
            point, direction = res
            pair_lines.append(((i, j), point, direction))
    if len(pair_lines) < 2:
        raise ValueError("not enough pair-lines to generate plane candidates")

    view_dirs = {
        idx: view_forward_direction(images_sent[idx]["R"], images_sent[idx]["T"], fov_deg, image_size)
        for idx in view_idxs
    }

    def score_normal(n: np.ndarray) -> float:
        vals = sorted(abs(np.dot(view_dirs[i], n)) for i in view_idxs)
        k = max(2, len(vals) // 2)
        return float(np.mean(vals[:k]))

    candidates = []
    for a in range(len(pair_lines)):
        pa, _, dir_a = pair_lines[a]
        for b in range(a + 1, len(pair_lines)):
            pb, _, dir_b = pair_lines[b]
            if set(pa) & set(pb):
                continue
            n = np.cross(dir_a, dir_b)
            norm = np.linalg.norm(n)
            if norm < 1e-6:
                continue
            candidates.append(n / norm)
    if not candidates:
        raise ValueError("could not generate any candidate normal (pair-lines too parallel)")

    scored = sorted(candidates, key=score_normal)
    best_normal = scored[0]
    good_views = [i for i in view_idxs if abs(np.dot(view_dirs[i], best_normal)) < edge_on_thresh]

    refit_dirs   = [d for (pa, _, d) in pair_lines if set(pa).issubset(good_views)]
    refit_points = [p for (pa, p, _) in pair_lines if set(pa).issubset(good_views)]

    if len(refit_dirs) >= 2:
        D = np.asarray(refit_dirs)
        _, _, Vt = np.linalg.svd(D)
        refined_normal = Vt[-1]
        refined_normal /= np.linalg.norm(refined_normal)
        used_points = refit_points
    else:
        refined_normal = best_normal
        used_points = [p for (_, p, _) in pair_lines]

    origin = np.mean(used_points, axis=0)
    return {
        "normal": refined_normal.tolist(), "origin": origin.tolist(),
        "n_views_used": len(good_views), "n_candidates": len(candidates), "good_views": good_views,
    }


def detect_planes_no_mesh(points_by_image: dict, images_sent: list, fov_deg: float, image_size: int,
                          edge_on_thresh: float = EDGE_ON_THRESH_DEFAULT, max_planes: int = 1,
                          dup_angle_thresh_deg: float = DUP_ANGLE_THRESH_DEFAULT) -> list:
    """Encuentra el plano 1, remueve del pool las vistas 'de canto' que lo
    apoyaron, repite sobre el resto para plano 2, 3, etc."""
    def _ang(v1, v2):
        v1, v2 = v1 / np.linalg.norm(v1), v2 / np.linalg.norm(v2)
        return float(np.degrees(np.arccos(np.clip(np.abs(np.dot(v1, v2)), 0.0, 1.0))))

    all_view_idxs = sorted(
        int(k) for k, pts in points_by_image.items()
        if get_point_by_obj_id(pts, 1) is not None and get_point_by_obj_id(pts, 2) is not None
    )
    pool = set(all_view_idxs)
    planes = []
    while len(planes) < max_planes:
        if len(pool) < 4:
            break
        sub_points = {k: v for k, v in points_by_image.items() if int(k) in pool}
        try:
            pred = estimate_plane_no_mesh(sub_points, images_sent, fov_deg, image_size, edge_on_thresh)
        except ValueError:
            break
        normal = np.array(pred["normal"])
        if any(_ang(normal, np.array(p["normal"])) < dup_angle_thresh_deg for p in planes):
            break
        planes.append(pred)
        pool -= set(pred["good_views"])
    return planes

## 5. Molmo2: carga del modelo + inferencia (inline) -- necesita GPU

Copia funcional de `MolmoPointing/molmo_multiview_runner.py` (Flow A). Sirve
igual para eje y plano -- no depende del symmetry_type, solo de
imagenes+prompt.

In [9]:
_processor = None
_model     = None


def get_model():
    global _processor, _model
    if _processor is None or _model is None:
        print(f"[model] Loading {MODEL_ID} ...")
        _processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, device_map="auto", use_fast=True)
        _model = AutoModelForImageTextToText.from_pretrained(
            MODEL_ID, trust_remote_code=True, device_map="auto", dtype=torch.bfloat16,
        )
        _model.eval()
        print("[model] Ready.")
    return _processor, _model


def call_model(images: list, prompt: str) -> str:
    processor, model = get_model()
    content = [{"type": "text", "text": prompt}]
    for img in images:
        content.append({"type": "image", "image": img})
    messages = [{"role": "user", "content": content}]
    inputs = processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt", return_dict=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.inference_mode():
        output_ids = model.generate(**inputs, max_new_tokens=2048)
    n_input = inputs["input_ids"].size(1)
    del inputs
    text = processor.tokenizer.decode(output_ids[0, n_input:], skip_special_tokens=True)
    del output_ids
    torch.cuda.empty_cache()
    return text


def parse_single_coords(text: str) -> dict:
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    raw = [float(n) for n in match.group(1).split()]
    if len(raw) < 4:
        return {}
    raw = raw[1:]
    pts = [{"obj_id": int(raw[i]), "x": raw[i + 1], "y": raw[i + 2]} for i in range(0, len(raw) - 2, 3)]
    return {"0": pts} if pts else {}


def parse_multi_coords(text: str, n_images: int) -> dict:
    match = re.search(r'coords=["\']([^"\']+)["\']', text)
    if not match:
        return {}
    result: dict = {}
    for group in match.group(1).split(";"):
        group = group.strip()
        if not group:
            continue
        nums = group.split()
        if len(nums) < 4:
            continue
        try:
            img_idx = int(float(nums[0])) - 1
        except ValueError:
            continue
        if img_idx < 0 or img_idx >= n_images:
            continue
        key, rest, pts = str(img_idx), nums[1:], []
        for i in range(0, len(rest) - 2, 3):
            try:
                pts.append({"obj_id": int(float(rest[i])), "x": float(rest[i + 1]), "y": float(rest[i + 2])})
            except ValueError:
                continue
        if pts:
            result.setdefault(key, []).extend(pts)
    return result


def load_metadata(render_dir: Path) -> list:
    path = render_dir / "metadata_all.json"
    if not path.exists():
        raise FileNotFoundError(f"metadata_all.json not found: {render_dir}")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def get_n_views_entries(metadata: list, n_views: int) -> list:
    total = len(metadata)
    if n_views >= total:
        return sorted(metadata, key=lambda e: e["index"])
    indices = {int(round(i)) for i in np.linspace(0, total - 1, n_views)}
    entries = [m for m in metadata if m["index"] in indices]
    return sorted(entries, key=lambda e: e["index"])


def run_inference(images: list, prompt_single: str, prompt_multi: str):
    """auto mode: 1 imagen -> prompt_single; >1 -> prompt_multi (1 solo llamado)."""
    if len(images) == 1:
        raw = call_model(images, prompt_single)
        pts = parse_single_coords(raw)
        return raw, ({"0": pts["0"]} if "0" in pts else {})
    raw = call_model(images, prompt_multi)
    return raw, parse_multi_coords(raw, n_images=len(images))

## 6. Correr inferencia (eje y plano, por separado)

Guarda `molmo_multiview_<EXPERIMENT_ID>.json` dentro del sandbox. Si el JSON
de una vista ya existe, se saltea -- borralo (o cambia el EXPERIMENT_ID) para
forzar recalculo.

In [10]:
def run_inference_for_objects(symmetry_type: str, object_list: list, experiment_id: str,
                              prompt_single: str, prompt_multi: str) -> None:
    for cat, oid in object_list:
        render_dir = SANDBOX_RENDERS / symmetry_type / oid / str(SIZE) / LIGHTING
        metadata   = load_metadata(render_dir)
        json_path  = render_dir / f"molmo_multiview_{experiment_id}.json"
        results    = json.load(open(json_path, encoding="utf-8")) if json_path.exists() else {}

        for n_views in VIEW_GROUPS:
            if str(n_views) in results:
                continue
            entries = get_n_views_entries(metadata, n_views)
            images  = [Image.open(render_dir / e["filename"]).convert("RGB") for e in entries]
            raw, points_by_img = run_inference(images, prompt_single, prompt_multi)

            results[str(n_views)] = {
                "experiment_id": experiment_id,
                "prompt_used": prompt_single if n_views == 1 else prompt_multi,
                "raw_output": raw, "points_by_image": points_by_img,
                "images_sent": [
                    {"img_idx": i, "filename": e["filename"], "index": e["index"],
                     "azimuth": e["azimuth"], "elevation": e["elevation"], "eye": e["eye"],
                     "R": e["R"], "T": e["T"]}
                    for i, e in enumerate(entries)
                ],
                "n_points": sum(len(v) for v in points_by_img.values()),
            }
            json_path.parent.mkdir(parents=True, exist_ok=True)
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(results, f, indent=2)
            print(f"  [{symmetry_type:9s}][{cat:6s}] {oid} n_views={n_views}: "
                  f"{results[str(n_views)]['n_points']} puntos devueltos")

In [11]:
# ── Celda 18 — Inferencia eje (con pre-check completo) ──────────────────────
def all_predictions_exist(symmetry_type: str, object_list: list,
                           experiment_id: str, view_groups: list) -> bool:
    """Devuelve True si TODOS los (objeto × n_views) ya tienen predicción guardada.
    Si es True, se puede saltar la carga del modelo por completo."""
    for _, oid in object_list:
        json_path = (SANDBOX_RENDERS / symmetry_type / oid
                     / str(SIZE) / LIGHTING
                     / f"molmo_multiview_{experiment_id}.json")
        if not json_path.exists():
            return False
        with open(json_path, encoding="utf-8") as f:
            results = json.load(f)
        if any(str(nv) not in results for nv in view_groups):
            return False
    return True


EXPERIMENT_ID_AXIS = "axis_v06_sandbox"

if all_predictions_exist("axis_sym", OBJECT_LIST_AXIS, EXPERIMENT_ID_AXIS, VIEW_GROUPS):
    print(f"[eje] Todas las predicciones ya existen para '{EXPERIMENT_ID_AXIS}' — modelo NO cargado.")
else:
    run_inference_for_objects("axis_sym", OBJECT_LIST_AXIS, EXPERIMENT_ID_AXIS,
                              PROMPT_SINGLE_AXIS, PROMPT_MULTI_AXIS)
    print("Inferencia de eje completa.")

[model] Loading allenai/Molmo2-8B ...


`use_fast` is set to `True` but the image processor class does not have a fast version.  Falling back to the slow version.


Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

[model] Ready.
  [axis_sym ][BUENO ] 941271c5d9b192eaccd8f9b9403fd602 n_views=6: 12 puntos devueltos
  [axis_sym ][BUENO ] 941271c5d9b192eaccd8f9b9403fd602 n_views=14: 28 puntos devueltos
  [axis_sym ][BUENO ] 941271c5d9b192eaccd8f9b9403fd602 n_views=26: 52 puntos devueltos
  [axis_sym ][BUENO ] e0725fd7859fa238ff67c12005f72d2 n_views=6: 12 puntos devueltos
  [axis_sym ][BUENO ] e0725fd7859fa238ff67c12005f72d2 n_views=14: 28 puntos devueltos
  [axis_sym ][BUENO ] e0725fd7859fa238ff67c12005f72d2 n_views=26: 52 puntos devueltos
  [axis_sym ][BUENO ] 89cb9b2ad175b833cadf6344ec272e8 n_views=6: 12 puntos devueltos
  [axis_sym ][BUENO ] 89cb9b2ad175b833cadf6344ec272e8 n_views=14: 28 puntos devueltos
  [axis_sym ][BUENO ] 89cb9b2ad175b833cadf6344ec272e8 n_views=26: 52 puntos devueltos
  [axis_sym ][BUENO ] dc2c07ff617d1da0197a35146ee825cd n_views=6: 12 puntos devueltos
  [axis_sym ][BUENO ] dc2c07ff617d1da0197a35146ee825cd n_views=14: 28 puntos devueltos
  [axis_sym ][BUENO ] dc2c07ff617d1da0

In [12]:
# ── Celda 19 — Inferencia plano (con pre-check completo) ────────────────────
EXPERIMENT_ID_PLANE = "plane_v04_1_sandbox"

if all_predictions_exist("plane_sym", OBJECT_LIST_PLANE, EXPERIMENT_ID_PLANE, VIEW_GROUPS):
    print(f"[plano] Todas las predicciones ya existen para '{EXPERIMENT_ID_PLANE}' — modelo NO cargado.")
else:
    run_inference_for_objects("plane_sym", OBJECT_LIST_PLANE, EXPERIMENT_ID_PLANE,
                              PROMPT_SINGLE_PLANE, PROMPT_MULTI_PLANE)
    print("Inferencia de plano completa.")

  [plane_sym][BUENO ] 109a19840a8fbc774c3aee8e9d3a6ffa n_views=6: 12 puntos devueltos
  [plane_sym][BUENO ] 109a19840a8fbc774c3aee8e9d3a6ffa n_views=14: 28 puntos devueltos
  [plane_sym][BUENO ] 109a19840a8fbc774c3aee8e9d3a6ffa n_views=26: 52 puntos devueltos
  [plane_sym][BUENO ] 1541e36e8dc2d84caed2201239784a35 n_views=6: 12 puntos devueltos
  [plane_sym][BUENO ] 1541e36e8dc2d84caed2201239784a35 n_views=14: 28 puntos devueltos
  [plane_sym][BUENO ] 1541e36e8dc2d84caed2201239784a35 n_views=26: 52 puntos devueltos
  [plane_sym][BUENO ] 15e2d1fe9a4b28e68fb8c03013603b0c n_views=6: 12 puntos devueltos
  [plane_sym][BUENO ] 15e2d1fe9a4b28e68fb8c03013603b0c n_views=14: 28 puntos devueltos
  [plane_sym][BUENO ] 15e2d1fe9a4b28e68fb8c03013603b0c n_views=26: 52 puntos devueltos
  [plane_sym][BUENO ] 1655608678a4f9ffbc7bb5239e53ea6f n_views=6: 12 puntos devueltos
  [plane_sym][BUENO ] 1655608678a4f9ffbc7bb5239e53ea6f n_views=14: 28 puntos devueltos
  [plane_sym][BUENO ] 1655608678a4f9ffbc7bb5239

## 7. Estimacion sin malla (eje: `triangulation`; plano: `triangulation_multiplane`)

In [13]:
FILTER_OFF_OBJECT    = False   # True para probar el filtro fondo/objeto (aplica a ambos)
MIN_POINTS_ON_OBJECT = 2

predicted_axes = {}   # (object_id, n_views) -> {"direction", "origin"} | None

for cat, oid in OBJECT_LIST_AXIS:
    render_dir = SANDBOX_RENDERS / "axis_sym" / oid / str(SIZE) / LIGHTING
    manifest_p = render_dir / "manifest.json"
    manifest   = json.load(open(manifest_p, encoding="utf-8")) if manifest_p.exists() else {}
    fov_deg, image_size = manifest.get("fov", DEFAULT_FOV), manifest.get("image_size", SIZE)

    molmo_data  = json.load(open(render_dir / f"molmo_multiview_{EXPERIMENT_ID_AXIS}.json", encoding="utf-8"))
    image_cache = {}
    for n_views_key, group in molmo_data.items():
        points_by_image, images_sent = group["points_by_image"], group["images_sent"]
        if FILTER_OFF_OBJECT:
            points_by_image = filter_points_on_object(
                points_by_image, images_sent, render_dir,
                min_points=MIN_POINTS_ON_OBJECT, image_cache=image_cache,
            )
        try:
            pred = estimate_axis_no_mesh(points_by_image, images_sent, fov_deg, image_size)
        except ValueError as e:
            pred = None
            print(f"  [{cat:6s}] {oid} n_views={n_views_key}: [omitido] {e}")
        predicted_axes[(oid, int(n_views_key))] = pred

print(f"\n[eje] {sum(v is not None for v in predicted_axes.values())}/{len(predicted_axes)} predicciones validas.")

  [MALO  ] 6e1fe96adbb5ffba8bae2d07dadd1b5d n_views=6: [omitido] need >=2 valid views, got 0
  [MALO  ] 56efabe64ce30e08d7bc2c778b186f58 n_views=26: [omitido] need >=2 valid views, got 0

[eje] 88/90 predicciones validas.


In [14]:
MAX_PLANES = 3   # igual convencion que --max-planes en estimate_symmetry_no_mesh.py

predicted_planes = {}   # (object_id, n_views) -> lista de planos | []

for cat, oid in OBJECT_LIST_PLANE:
    render_dir = SANDBOX_RENDERS / "plane_sym" / oid / str(SIZE) / LIGHTING
    manifest_p = render_dir / "manifest.json"
    manifest   = json.load(open(manifest_p, encoding="utf-8")) if manifest_p.exists() else {}
    fov_deg, image_size = manifest.get("fov", DEFAULT_FOV), manifest.get("image_size", SIZE)

    molmo_data  = json.load(open(render_dir / f"molmo_multiview_{EXPERIMENT_ID_PLANE}.json", encoding="utf-8"))
    image_cache = {}
    for n_views_key, group in molmo_data.items():
        points_by_image, images_sent = group["points_by_image"], group["images_sent"]
        if FILTER_OFF_OBJECT:
            points_by_image = filter_points_on_object(
                points_by_image, images_sent, render_dir,
                min_points=MIN_POINTS_ON_OBJECT, image_cache=image_cache,
            )
        planes = detect_planes_no_mesh(
            points_by_image, images_sent, fov_deg, image_size,
            edge_on_thresh=EDGE_ON_THRESH_DEFAULT, max_planes=MAX_PLANES,
            dup_angle_thresh_deg=DUP_ANGLE_THRESH_DEFAULT,
        )
        if not planes:
            print(f"  [{cat:6s}] {oid} n_views={n_views_key}: [omitido] no se detecto ningun plano")
        predicted_planes[(oid, int(n_views_key))] = planes

print(f"\n[plano] {sum(len(v) > 0 for v in predicted_planes.values())}/{len(predicted_planes)} objetos con >=1 plano detectado.")

  [BUENO ] 1ad4c572e0fd6a576e1e9a13188ab4bb n_views=26: [omitido] no se detecto ningun plano
  [MEDIO ] 82d8391c62e197161282d4d9178caa92 n_views=26: [omitido] no se detecto ningun plano
  [MEDIO ] 835e01af50296235aefda81565fede7 n_views=6: [omitido] no se detecto ningun plano
  [MEDIO ] 8399987b9bca0dd0e63a9e8397b31118 n_views=6: [omitido] no se detecto ningun plano
  [MEDIO ] 8497e02fa1662113776d8bc79b9caa2c n_views=6: [omitido] no se detecto ningun plano
  [MEDIO ] 84aa911799cd87b4ad5067eac75a07f7 n_views=6: [omitido] no se detecto ningun plano
  [MALO  ] a85b9a3ace923e424721d5612f98ae26 n_views=26: [omitido] no se detecto ningun plano
  [MALO  ] ad86354fb5faf1c98a4820926b2a786 n_views=26: [omitido] no se detecto ningun plano
  [MALO  ] d2553e5fc4f1527cfeae521e94848af6 n_views=26: [omitido] no se detecto ningun plano
  [MALO  ] dd2a4c416625f29c4f57a7ededfb3bde n_views=26: [omitido] no se detecto ningun plano

[plano] 80/90 objetos con >=1 plano detectado.


## 8. Evaluacion contra GT (eje y plano)

Copia funcional de `Mapping/evaluate.py::parse_true_label` /
`angular_error_deg` / `point_to_line_distance` / `evaluate_plane_multiset`.

In [15]:
ANGULAR_THRESHOLDS = [5, 10, 15]


def parse_true_label(txt_path: Path) -> dict:
    """Devuelve {"type": "axis"|"plane", "elements": [...]} -- una malla puede
    tener MAS DE UN plano GT (n_true_planes_mean ~1.15-1.2 en el sweep real)."""
    lines, elements, sym_type = [l.strip() for l in txt_path.read_text().splitlines() if l.strip()], [], None
    for line in lines:
        if line.startswith("axis"):
            sym_type = "axis"
            parts = line.split()
            vec  = np.array([float(x) for x in parts[1:4]]); vec /= np.linalg.norm(vec)
            elements.append({"direction": vec.tolist(), "origin": [float(x) for x in parts[4:7]]})
        elif line.startswith("plane"):
            sym_type = "plane"
            parts = line.split()
            vec  = np.array([float(x) for x in parts[1:4]]); vec /= np.linalg.norm(vec)
            elements.append({"normal": vec.tolist(), "origin": [float(x) for x in parts[4:7]]})
    return {"type": sym_type, "elements": elements}


def angular_error_deg(v1: np.ndarray, v2: np.ndarray) -> float:
    v1, v2 = v1 / np.linalg.norm(v1), v2 / np.linalg.norm(v2)
    return float(np.degrees(np.arccos(np.clip(np.abs(np.dot(v1, v2)), 0.0, 1.0))))


def point_to_line_distance(point: np.ndarray, line_origin: np.ndarray, line_dir: np.ndarray) -> float:
    d = line_dir / np.linalg.norm(line_dir)
    v = point - line_origin
    return float(np.linalg.norm(v - np.dot(v, d) * d))


def evaluate_plane_multiset(pred_planes: list, true_elements: list, angular_threshold_deg: float = 15.0) -> dict:
    """Greedy best-match recall/precision sobre el CONJUNTO COMPLETO de planos GT
    -- Mapping/evaluate.py::evaluate_plane_multiset, copia exacta."""
    n_pred, n_true, matched_gt = len(pred_planes), len(true_elements), set()
    for pred in pred_planes:
        p_normal = np.array(pred["normal"])
        best_idx, best_ang = -1, float("inf")
        for idx, true in enumerate(true_elements):
            if idx in matched_gt:
                continue
            ang = angular_error_deg(p_normal, np.array(true["normal"]))
            if ang < best_ang:
                best_idx, best_ang = idx, ang
        if best_idx >= 0 and best_ang < angular_threshold_deg:
            matched_gt.add(best_idx)
    n_matched = len(matched_gt)
    return {
        "n_planes_predicted": n_pred, "n_true_planes": n_true, "n_planes_matched": n_matched,
        "recall_planes":    round(n_matched / n_true, 4) if n_true else None,
        "precision_planes": round(n_matched / n_pred, 4) if n_pred else None,
    }

In [16]:
# --- SDE_ref / F1_ref (Mapping/evaluate.py, formulas verbatim) ---
# Requiere gpytoolbox (superficie real via AABB tree) -- disponible en el servidor.
import gpytoolbox as gpy

THRESHOLDS_INLIER    = [0.05, 0.1, 0.15, 0.2]   # metric_F1.py's set_threshold
N_SAMPLES_DEFAULT    = 1000                      # metric_SDE.py's sample count
SDE_REF_SEED_DEFAULT = 0                         # reproducible surface sampling


def sample_surface_points(mesh, n_samples: int = N_SAMPLES_DEFAULT, seed: int | None = SDE_REF_SEED_DEFAULT):
    """Muestra area-weighted de la superficie real -- usada por SDE_ref (eje y plano)."""
    v = np.asarray(mesh.vertices, dtype=np.float64)
    f = np.asarray(mesh.faces, dtype=np.int64)
    if seed is not None:
        np.random.seed(seed)  # el sampler de gpytoolbox usa el RNG global de numpy
    sample = gpy.random_points_on_mesh(v, f, n_samples)
    return v, f, sample


def calaxisloss(axis_dir: np.ndarray, axis_origin: np.ndarray, vertices: np.ndarray,
                faces: np.ndarray, points: np.ndarray) -> float:
    """SDE_ref para eje: refleja puntos de la superficie real 180 grados sobre
    el eje PREDICHO y mide la distancia (al cuadrado) a la superficie real mas
    cercana -- NO usa el GT, es autoconsistencia pura (self-supervised)."""
    t = (points - axis_origin) @ axis_dir
    proj = axis_origin + t[:, None] * axis_dir
    reflected = 2.0 * proj - points
    d, ind, b = gpy.squared_distance(reflected, vertices, faces, use_aabb=True, use_cpp=True)
    return float(np.mean(d))


def calplaneloss(plane: np.ndarray, vertices: np.ndarray, faces: np.ndarray, points: np.ndarray) -> float:
    """SDE_ref para plano: mismo principio, reflejo especular sobre el plano PREDICHO."""
    points = np.hstack((points, np.ones((points.shape[0], 1))))
    lam = points.dot(plane.T)
    planepoints = points - 2 * lam * plane
    d, ind, b = gpy.squared_distance(planepoints[:, 0:3], vertices, faces, use_aabb=True, use_cpp=True)
    return float(np.mean(d))


def normal_origin_to_plane(normal: list, origin: list) -> np.ndarray:
    """Convencion [nx, ny, nz, d] con d = -origin . normal -- la que usa F1_ref."""
    normal = np.asarray(normal, dtype=np.float64)
    origin = np.asarray(origin, dtype=np.float64)
    d = -origin.dot(normal)
    return np.array([normal[0], normal[1], normal[2], d]).reshape(1, 4)


def f1_match_counts(predicted: list, gt_planes: list, threshold_inlier: float) -> tuple[int, int, int]:
    """Copia verbatim de Mapping/evaluate.py::f1_match_counts (greedy, en orden
    de lista -- convencion PRS-Net/E3Sym, NO asignacion optima). El conteo de
    fp por cada gt no-matcheado dentro del loop interno es un detalle real del
    metodo de referencia -- no se "limpia", se replica tal cual."""
    mask = np.zeros(len(gt_planes), dtype=bool)
    tp = fp = 0
    for pred_plane in predicted:
        for idx, gt in enumerate(gt_planes):
            if mask[idx]:
                continue
            val1 = np.linalg.norm(pred_plane - gt)
            val2 = np.linalg.norm(pred_plane + gt)
            val = min(val1, val2)
            if val < threshold_inlier:
                mask[idx] = True
                tp += 1
            else:
                fp += 1
    fn = int(np.sum(~mask))
    return tp, fp, fn


def f1_from_counts(tp: int, fp: int, fn: int) -> float:
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall    = tp / (tp + fn) if (tp + fn) else 0.0
    return 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0


print("Nota: F1_ref NO existe para axis_sym (ni el repo de referencia ni la "
      "literatura tienen una convencion de F1 para ejes) -- solo se reporta SDE_ref.")

Nota: F1_ref NO existe para axis_sym (ni el repo de referencia ni la literatura tienen una convencion de F1 para ejes) -- solo se reporta SDE_ref.


In [17]:
mesh_cache_axis = {}   # object_id -> (mesh_v, mesh_f, sample) -- se reusa entre n_views

rows_axis = []
for cat, oid in OBJECT_LIST_AXIS:
    gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.txt")
    t_dir, t_orig = np.array(gt["elements"][0]["direction"]), np.array(gt["elements"][0]["origin"])

    if oid not in mesh_cache_axis:
        mesh = trimesh.load(str(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.obj"), force="mesh", process=False)
        mesh_cache_axis[oid] = sample_surface_points(mesh)
    mesh_v, mesh_f, sample = mesh_cache_axis[oid]

    for n_views in VIEW_GROUPS:
        pred = predicted_axes.get((oid, n_views))
        row = {"categoria": cat, "object_id": oid, "n_views": n_views}
        if pred is None:
            row.update({"status": "no_pred", "angular_error_deg": 90.0, "translation_error": None, "sde_ref": None})
        else:
            p_dir, p_orig = np.array(pred["direction"]), np.array(pred["origin"])
            ang, dist = angular_error_deg(p_dir, t_dir), point_to_line_distance(p_orig, t_orig, t_dir)
            sde = calaxisloss(p_dir, p_orig, mesh_v, mesh_f, sample)
            row.update({"status": "ok", "angular_error_deg": round(ang, 4), "translation_error": round(dist, 6),
                        "sde_ref": round(sde, 8)})
            for t in ANGULAR_THRESHOLDS:
                row[f"precision_{t}deg"] = int(ang < t)
        rows_axis.append(row)

df_eval_axis = pd.DataFrame(rows_axis).sort_values(["categoria", "object_id", "n_views"]).reset_index(drop=True)
df_eval_axis

,categoria,object_id,n_views,status,angular_error_deg,translation_error,sde_ref,precision_5deg,precision_10deg,precision_15deg
0,BUENO,28d39c9673775f601a3b47d6d0918049,6,ok,72.3390,1.227088,0.007962,0.0,0.0,0.0
1,BUENO,28d39c9673775f601a3b47d6d0918049,14,ok,58.9223,0.183089,0.032244,0.0,0.0,0.0
2,BUENO,28d39c9673775f601a3b47d6d0918049,26,ok,3.1327,0.072416,0.004414,1.0,1.0,1.0
3,BUENO,3a29634236aa0596f9e8cd846ef13776,6,ok,28.6762,0.011905,0.049414,0.0,0.0,0.0
4,BUENO,3a29634236aa0596f9e8cd846ef13776,14,ok,11.0314,0.008093,0.007582,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...
85,MEDIO,d851cbc873de1c4d3b6eb309177a6753,14,ok,54.6051,0.032048,0.027043,0.0,0.0,0.0
86,MEDIO,d851cbc873de1c4d3b6eb309177a6753,26,ok,58.3236,0.035552,0.025250,0.0,0.0,0.0
87,MEDIO,f0611ec9ff89209bf10c4513652c1c5e,6,ok,22.9296,0.009383,0.021323,0.0,0.0,0.0
88,MEDIO,f0611ec9ff89209bf10c4513652c1c5e,14,ok,49.1808,0.006380,0.044274,0.0,0.0,0.0


In [18]:
mesh_cache_plane = {}   # object_id -> (mesh_v, mesh_f, sample, gt_planes_ref)

rows_plane = []
for cat, oid in OBJECT_LIST_PLANE:
    gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["plane_sym"] / f"{oid}.txt")
    true_elements = gt["elements"]

    if oid not in mesh_cache_plane:
        mesh = trimesh.load(str(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["plane_sym"] / f"{oid}.obj"), force="mesh", process=False)
        mesh_v, mesh_f, sample = sample_surface_points(mesh)
        gt_planes_ref = [normal_origin_to_plane(e["normal"], e["origin"]) for e in true_elements]
        mesh_cache_plane[oid] = (mesh_v, mesh_f, sample, gt_planes_ref)
    mesh_v, mesh_f, sample, gt_planes_ref = mesh_cache_plane[oid]

    for n_views in VIEW_GROUPS:
        planes = predicted_planes.get((oid, n_views), [])
        row = {"categoria": cat, "object_id": oid, "n_views": n_views, "n_true_planes": len(true_elements)}
        if not planes:
            row.update({"status": "no_pred", "recall_planes": 0.0, "precision_planes": None,
                        "n_planes_predicted": 0, "sde_ref": None, "f1_counts": None})
        else:
            m = evaluate_plane_multiset(planes, true_elements, angular_threshold_deg=ANGULAR_THRESHOLDS[-1])
            pred_planes_ref = [normal_origin_to_plane(p["normal"], p["origin"]) for p in planes]
            sde_per_plane = [calplaneloss(pp, mesh_v, mesh_f, sample) for pp in pred_planes_ref]
            f1_counts = {t: f1_match_counts(pred_planes_ref, gt_planes_ref, t) for t in THRESHOLDS_INLIER}
            row.update({"status": "ok", **m,
                        "sde_ref": round(float(np.mean(sde_per_plane)), 8),
                        "f1_counts": f1_counts})
        rows_plane.append(row)

df_eval_plane = pd.DataFrame(rows_plane).sort_values(["categoria", "object_id", "n_views"]).reset_index(drop=True)
# 'f1_counts' es un dict crudo (para agregar F1_ref en la celda siguiente) -- se oculta solo para mostrar la tabla:
df_eval_plane.drop(columns=["f1_counts"])

/home/pespinoz/miniconda3/envs/tesis_env/lib/python3.10/site-packages/gpytoolbox/barycentric_coordinates.py:104: RuntimeWarning: invalid value encountered in divide
  gamma = np.sum(np.multiply(np.cross(u,w,axis=1),N),axis=1)/n2
/home/pespinoz/miniconda3/envs/tesis_env/lib/python3.10/site-packages/gpytoolbox/barycentric_coordinates.py:105: RuntimeWarning: invalid value encountered in divide
  beta = np.sum(np.multiply(np.cross(w,v,axis=1),N),axis=1)/n2


,categoria,object_id,n_views,n_true_planes,status,n_planes_predicted,n_planes_matched,recall_planes,precision_planes,sde_ref
0,BUENO,109a19840a8fbc774c3aee8e9d3a6ffa,6,1,ok,1,0.0,0.0,0.0000,0.007564
1,BUENO,109a19840a8fbc774c3aee8e9d3a6ffa,14,1,ok,1,1.0,1.0,1.0000,0.000129
2,BUENO,109a19840a8fbc774c3aee8e9d3a6ffa,26,1,ok,3,1.0,1.0,0.3333,0.006827
3,BUENO,1541e36e8dc2d84caed2201239784a35,6,1,ok,1,0.0,0.0,0.0000,0.035129
4,BUENO,1541e36e8dc2d84caed2201239784a35,14,1,ok,1,1.0,1.0,1.0000,0.000717
...,...,...,...,...,...,...,...,...,...,...
85,MEDIO,854ffe2c60091876e07a1c4b84dfd325,14,1,ok,2,1.0,1.0,0.5000,0.017334
86,MEDIO,854ffe2c60091876e07a1c4b84dfd325,26,1,ok,3,1.0,1.0,0.3333,0.016894
87,MEDIO,8596664f3d7925cdfdeb515ad63cf4b0,6,1,ok,1,1.0,1.0,1.0000,0.000518
88,MEDIO,8596664f3d7925cdfdeb515ad63cf4b0,14,1,ok,2,1.0,1.0,0.5000,0.011986


### 8.1 SDE_ref / F1_ref por conjunto y subconjunto (BUENO/MEDIO/MALO)

`SDE_ref` (eje y plano) se promedia simple dentro de cada grupo. `F1_ref`
(solo plano) acumula TP/FP/FN de TODOS los objetos del grupo por umbral
ANTES de calcular F1 -- no es el promedio de un F1 por objeto (misma
convencion que `Mapping/evaluate.py::_f1_from_counts_by_threshold`; ver
`docs/diagnostico_conditioning_axis.md` para por que esto importa).

In [19]:
def aggregate_group_metrics(df_axis: pd.DataFrame, df_plane: pd.DataFrame):
    grupos = ["TODOS", "BUENO", "MEDIO", "MALO"]

    axis_rows = []
    for n_views in VIEW_GROUPS:
        for grupo in grupos:
            sub = df_axis[df_axis["n_views"] == n_views]
            if grupo != "TODOS":
                sub = sub[sub["categoria"] == grupo]
            sde_vals = sub["sde_ref"].dropna()
            axis_rows.append({
                "grupo": grupo, "n_views": n_views, "n_objects": len(sub),
                "sde_ref_mean": round(float(sde_vals.mean()), 8) if len(sde_vals) else None,
            })

    plane_rows = []
    for n_views in VIEW_GROUPS:
        for grupo in grupos:
            sub = df_plane[df_plane["n_views"] == n_views]
            if grupo != "TODOS":
                sub = sub[sub["categoria"] == grupo]
            sde_vals = sub["sde_ref"].dropna()

            totals = {t: [0, 0, 0] for t in THRESHOLDS_INLIER}
            for counts in sub["f1_counts"].dropna():
                for t in THRESHOLDS_INLIER:
                    tp, fp, fn = counts[t]
                    totals[t][0] += tp; totals[t][1] += fp; totals[t][2] += fn
            has_any = any(sum(totals[t]) > 0 for t in THRESHOLDS_INLIER)
            f1_per_t = [f1_from_counts(*totals[t]) for t in THRESHOLDS_INLIER]

            plane_rows.append({
                "grupo": grupo, "n_views": n_views, "n_objects": len(sub),
                "sde_ref_mean": round(float(sde_vals.mean()), 8) if len(sde_vals) else None,
                "f1_ref": round(float(np.mean(f1_per_t)), 4) if has_any else None,
            })

    return pd.DataFrame(axis_rows), pd.DataFrame(plane_rows)


axis_group_metrics, plane_group_metrics = aggregate_group_metrics(df_eval_axis, df_eval_plane)

print("=== SDE_ref por grupo x n_views -- EJE ===")
display(axis_group_metrics)
print("\n=== SDE_ref / F1_ref por grupo x n_views -- PLANO ===")
display(plane_group_metrics)

=== SDE_ref por grupo x n_views -- EJE ===


,grupo,n_views,n_objects,sde_ref_mean
0,TODOS,6,30,0.041766
1,BUENO,6,10,0.009351
2,MEDIO,6,10,0.044663
3,MALO,6,10,0.074563
4,TODOS,14,30,0.042836
5,BUENO,14,10,0.012407
6,MEDIO,14,10,0.050766
7,MALO,14,10,0.065335
8,TODOS,26,30,0.040939
9,BUENO,26,10,0.002632



=== SDE_ref / F1_ref por grupo x n_views -- PLANO ===


,grupo,n_views,n_objects,sde_ref_mean,f1_ref
0,TODOS,6,30,0.022472,0.1389
1,BUENO,6,10,0.019623,0.1000
2,MEDIO,6,10,0.014193,0.2083
3,MALO,6,10,0.030289,0.1364
4,TODOS,14,30,0.009462,0.3735
5,BUENO,14,10,0.001802,0.7250
6,MEDIO,14,10,0.008959,0.5767
7,MALO,14,10,0.017626,0.0000
8,TODOS,26,30,0.009926,0.2050
9,BUENO,26,10,0.011073,0.1290


## 9. Resultados: resumen por categoria y por n_views

---
# Experimentos de reducción de error angular — Literatura 2025-2026

Este notebook implementa tres intervenciones independientes derivadas de la
revisión de literatura (`propuestas_reduccion_error_angular.md`), evaluadas
sobre los mismos 30 objetos de `axis_sym` y los prompts `axis_v06_sandbox`.

Cada experimento es **independiente**: no requiere re-ejecutar los anteriores.
Todos reutilizan los datos de Molmo2 ya inferidos y las mallas de `mesh_cache_axis`.

| # | Nombre | Fuente | Mecanismo |
|---|--------|--------|-----------|
| **EXP-LIT-1** | Candidates-then-Select | CVPC arXiv:2512.04686; ZeroDex arXiv:2606.19340 | Reformula pointing como selección entre candidatos epipolares |
| **EXP-LIT-2** | Grid Overlay | MOKA arXiv:2403.03174; SketchVLM arXiv:2604.22875 | Superpone grilla de coordenadas visible sobre el render |
| **EXP-LIT-3** | Structured CoT | arXiv:2507.13362 | CoT tipo scene-graph antes de señalar coordenadas |

> **Referencia de baseline**: AUC@45° nv6=0.2195, nv14=0.1837, nv26=0.2882
> (axis_v06, 224px, sin gate, resultados_sandbox.md)


## 0. Helpers compartidos

In [ ]:
# ── Helpers de evaluación reutilizados por los tres experimentos ────────────
from PIL import Image, ImageDraw, ImageFont
import hashlib

def auc_angular(errors_deg: list, max_theta: float = 45.0) -> float:
    thetas = np.linspace(0, max_theta, 1000)
    errs   = np.array(errors_deg)
    precs  = [(errs <= t).mean() for t in thetas]
    return float(np.trapezoid(precs, thetas) / max_theta)


def eval_axis_rows(label: str, predictions: dict) -> pd.DataFrame:
    """
    Evalua un diccionario {(oid, n_views) -> pred | None} contra el GT de axis_sym.
    Devuelve DataFrame con las mismas columnas que df_eval_axis.
    """
    rows = []
    for cat, oid in OBJECT_LIST_AXIS:
        gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.txt")
        t_dir  = np.array(gt["elements"][0]["direction"])
        t_orig = np.array(gt["elements"][0]["origin"])
        if oid not in mesh_cache_axis:
            mesh = trimesh.load(str(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.obj"),
                                force="mesh", process=False)
            mesh_cache_axis[oid] = sample_surface_points(mesh)
        mesh_v, mesh_f, sample = mesh_cache_axis[oid]
        for n_views in VIEW_GROUPS:
            pred = predictions.get((oid, n_views))
            row  = {"experimento": label, "categoria": cat, "object_id": oid, "n_views": n_views}
            if pred is None:
                row.update({"angular_error_deg": 90.0, "sde_ref": None})
            else:
                p_dir  = np.array(pred["direction"])
                p_orig = np.array(pred["origin"])
                ang    = angular_error_deg(p_dir, t_dir)
                sde    = calaxisloss(p_dir, p_orig, mesh_v, mesh_f, sample)
                row.update({"angular_error_deg": round(ang, 4), "sde_ref": round(sde, 8)})
                for t in ANGULAR_THRESHOLDS:
                    row[f"precision_{t}deg"] = int(ang < t)
            rows.append(row)
    return pd.DataFrame(rows)


def summarize_experiment(label: str, df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["angular_error_deg"] = df["angular_error_deg"].fillna(90.0)
    summary = df.groupby("n_views")["angular_error_deg"].agg(
        mean="mean", median="median", std="std").round(2)
    for t in ANGULAR_THRESHOLDS:
        col = f"precision_{t}deg"
        if col in df.columns:
            summary[f"P@{t}°"] = df.groupby("n_views")[col].mean().round(4)
    summary["AUC@45°"] = {
        nv: round(auc_angular(df[df["n_views"]==nv]["angular_error_deg"].tolist()), 4)
        for nv in VIEW_GROUPS
    }
    if "sde_ref" in df.columns:
        summary["SDE_ref"] = df.groupby("n_views")["sde_ref"].mean().round(6)
    print(f"\n{'='*60}\n  {label}\n{'='*60}")
    print(summary.to_string())
    return summary


# Baseline para comparación (cargado de df_eval_axis ya calculado)
baseline_rows = []
for _, row in df_eval_axis.iterrows():
    baseline_rows.append({
        "experimento": "Baseline (axis_v06)",
        "categoria": row["categoria"],
        "object_id": row["object_id"],
        "n_views": row["n_views"],
        "angular_error_deg": row.get("angular_error_deg", 90.0),
        "sde_ref": row.get("sde_ref"),
        **{f"precision_{t}deg": row.get(f"precision_{t}deg", 0) for t in ANGULAR_THRESHOLDS},
    })
df_baseline = pd.DataFrame(baseline_rows)
summary_baseline = summarize_experiment("Baseline (axis_v06)", df_baseline)

print("\nHelpers listos. Ninguna celda de este bloque carga el modelo.")


---
## EXP-LIT-1 — Candidates-then-Select (selección entre candidatos epipolares)
*(CVPC arXiv:2512.04686; ZeroDex arXiv:2606.19340)*

**Motivación**: "Towards Cross-View Point Correspondence in VLMs" (CVPC, dic. 2025)
mide que Molmo-7B-D obtiene **3.02 %** de acierto en la tarea de predecir la
coordenada continua de un punto correspondiente en otra vista — prácticamente
azar. En cambio, en la tarea de *elegir entre 3 candidatos* (Correspondence-Judgement),
modelos comparables suben hasta 60-66 %. Brecha de ~45 pp.

**Estrategia** (inspirada en ZeroDex arXiv:2606.19340):

1. Vista de referencia (la más frontal al eje): Molmo señala el polo libremente
   → un polo de referencia por objeto.
2. Para cada vista siguiente: se proyecta el polo 3D (obtenido del baseline o de
   una estimación aproximada) al plano imagen de esa vista → punto central.
   Se generan **K=4 candidatos** a lo largo del rayo epipolar (±δ en X e Y).
3. Se marcan los candidatos con colores sobre la imagen y se le pregunta a Molmo
   **cuál corresponde al polo** → selección discreta.
4. El polo ganador de cada vista + el polo de referencia alimentan la triangulación.

**Implementación práctica**: como reutilizamos los datos de Molmo2 ya inferidos
(no podemos hacer nuevas inferencias aquí), simulamos el paso de selección usando
la proyección GT del polo para rankear cuál de los K candidatos generados alrededor
del punto predicho original es más cercano al GT proyectado. Esto mide el techo
del método (upper bound) — cuánto mejoraría si la selección fuera perfecta —
y el piso (lower bound) — si la selección es aleatoria entre candidatos.

Esta implementación permite **cuantificar la brecha** entre el baseline libre y
el upper/lower bound de la estrategia de candidatos.


In [ ]:
# ── EXP-LIT-1: Candidates-then-Select — upper/lower bound analysis ───────────
# Refs: CVPC arXiv:2512.04686; ZeroDex arXiv:2606.19340

K_CANDIDATES     = 4       # candidatos por vista (además del punto original)
DELTA_CANDIDATES = 30.0    # radio en px Molmo (0-1000) para generar candidatos
CANDIDATE_SEED   = 42

rng_cand = np.random.default_rng(CANDIDATE_SEED)


def project_3d_to_molmo(point_3d: np.ndarray, R: list, T: list,
                          fov_deg: float, image_size: int) -> tuple[float, float]:
    """Proyecta un punto 3D al espacio de coordenadas Molmo (0-1000)."""
    R_np, T_np = np.array(R, dtype=np.float64), np.array(T, dtype=np.float64)
    p_cam  = point_3d @ R_np + T_np
    half_tan = np.tan(np.deg2rad(fov_deg) / 2.0)
    # NDC (pinhole): X_ndc = x_cam / (z_cam * half_tan)
    if abs(p_cam[2]) < 1e-9:
        return 500.0, 500.0
    ndc_x =  p_cam[0] / (p_cam[2] * half_tan)
    ndc_y = -p_cam[1] / (p_cam[2] * half_tan)
    mx = (ndc_x + 1.0) / 2.0 * 1000.0
    my = (1.0 - (ndc_y + 1.0) / 2.0) * 1000.0
    return float(np.clip(mx, 0, 999)), float(np.clip(my, 0, 999))


def generate_candidates(center_x: float, center_y: float,
                         k: int = K_CANDIDATES, delta: float = DELTA_CANDIDATES,
                         rng: np.random.Generator | None = None) -> list[dict]:
    """Genera K candidatos en un disco de radio delta alrededor del punto central."""
    if rng is None:
        rng = np.random.default_rng(CANDIDATE_SEED)
    cands = [{"x": center_x, "y": center_y}]   # candidato 0 = punto original
    angles = rng.uniform(0, 2*np.pi, k)
    radii  = rng.uniform(delta*0.4, delta, k)
    for a, r in zip(angles, radii):
        cands.append({
            "x": float(np.clip(center_x + r * np.cos(a), 5, 995)),
            "y": float(np.clip(center_y + r * np.sin(a), 5, 995)),
        })
    return cands


def select_candidate(candidates: list[dict], gt_x: float, gt_y: float,
                      strategy: str = "oracle") -> dict:
    """
    Selecciona el candidato más cercano al GT proyectado (oracle/upper-bound)
    o uno aleatorio (random/lower-bound).
    strategy: 'oracle' | 'random' | 'original' (usa solo el punto del modelo)
    """
    if strategy == "oracle":
        dists = [((c["x"]-gt_x)**2 + (c["y"]-gt_y)**2)**0.5 for c in candidates]
        return candidates[int(np.argmin(dists))]
    elif strategy == "random":
        return candidates[rng_cand.integers(0, len(candidates))]
    else:  # original
        return candidates[0]


def run_candidates_experiment(strategy: str = "oracle") -> dict:
    """
    strategy: 'oracle' | 'random' | 'original'
    Devuelve {(oid, n_views) -> pred | None}
    """
    predictions = {}
    for cat, oid in OBJECT_LIST_AXIS:
        gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.txt")
        gt_dir  = np.array(gt["elements"][0]["direction"])
        gt_orig = np.array(gt["elements"][0]["origin"])

        render_dir = SANDBOX_RENDERS / "axis_sym" / oid / str(SIZE) / LIGHTING
        manifest_p = render_dir / "manifest.json"
        manifest   = json.load(open(manifest_p, encoding="utf-8")) if manifest_p.exists() else {}
        fov_deg    = manifest.get("fov", DEFAULT_FOV)
        image_size = manifest.get("image_size", SIZE)
        molmo_data = json.load(open(
            render_dir / f"molmo_multiview_{EXPERIMENT_ID_AXIS}.json", encoding="utf-8"))

        for n_views in VIEW_GROUPS:
            group      = molmo_data.get(str(n_views), {})
            pts_by_img = group.get("points_by_image", {})
            imgs_sent  = group.get("images_sent", [])

            # Punto GT en 3D para proyección por vista
            gt_point_top    = np.array(gt_orig) + 0.5 * gt_dir   # polo "superior" aprox
            gt_point_bottom = np.array(gt_orig) - 0.5 * gt_dir

            new_pts_by_img = {}
            for img_idx_str, pts in pts_by_img.items():
                pair = widest_pair(pts)
                if pair is None:
                    new_pts_by_img[img_idx_str] = []
                    continue
                p_a, p_b = pair
                cam = imgs_sent[int(img_idx_str)]
                R, T = cam["R"], cam["T"]

                # Proyección GT en esta vista
                gt_ax, gt_ay = project_3d_to_molmo(gt_point_top, R, T, fov_deg, image_size)
                gt_bx, gt_by = project_3d_to_molmo(gt_point_bottom, R, T, fov_deg, image_size)

                # Generar candidatos alrededor del punto predicho y seleccionar
                cands_a = generate_candidates(p_a["x"], p_a["y"])
                cands_b = generate_candidates(p_b["x"], p_b["y"])

                sel_a = select_candidate(cands_a, gt_ax, gt_ay, strategy)
                sel_b = select_candidate(cands_b, gt_bx, gt_by, strategy)

                new_pts_by_img[img_idx_str] = [
                    {"obj_id": 1, "x": sel_a["x"], "y": sel_a["y"]},
                    {"obj_id": 2, "x": sel_b["x"], "y": sel_b["y"]},
                ]

            # Triangular con los puntos seleccionados
            try:
                pred = estimate_axis_no_mesh(new_pts_by_img, imgs_sent, fov_deg, image_size)
            except ValueError:
                pred = None
            predictions[(oid, n_views)] = pred
    return predictions


# ── Correr las tres variantes ────────────────────────────────────────────────
print("EXP-LIT-1: Corriendo variantes...")

preds_oracle   = run_candidates_experiment("oracle")
preds_random   = run_candidates_experiment("random")
preds_original = run_candidates_experiment("original")   # debería replicar el baseline

df_oracle   = eval_axis_rows("EXP-LIT-1 Oracle (upper bound)",  preds_oracle)
df_random   = eval_axis_rows("EXP-LIT-1 Random (lower bound)",  preds_random)
df_original = eval_axis_rows("EXP-LIT-1 Original (control)",    preds_original)

s_oracle   = summarize_experiment("EXP-LIT-1 Oracle (upper bound)",  df_oracle)
s_random   = summarize_experiment("EXP-LIT-1 Random (lower bound)",  df_random)
s_original = summarize_experiment("EXP-LIT-1 Original (control)",    df_original)

# ── Tabla comparativa: baseline vs oracle vs random ─────────────────────────
print("\n=== Comparación EXP-LIT-1 — AUC@45° y error mediano ===")
comp_lit1 = pd.DataFrame({
    "Baseline":           {nv: round(auc_angular(df_baseline[df_baseline["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "Original (control)": {nv: round(auc_angular(df_original[df_original["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "Random (lb)":        {nv: round(auc_angular(df_random[df_random["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "Oracle (ub)":        {nv: round(auc_angular(df_oracle[df_oracle["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
}).rename_axis("n_views")
print(comp_lit1.to_string())

# Calcular brecha oracle-baseline y oracle-random
print("\n=== Brecha oracle vs baseline ===")
for nv in VIEW_GROUPS:
    auc_b = comp_lit1.loc[nv, "Baseline"]
    auc_o = comp_lit1.loc[nv, "Oracle (ub)"]
    auc_r = comp_lit1.loc[nv, "Random (lb)"]
    print(f"  nv={nv}: baseline={auc_b:.4f}  random={auc_r:.4f}  oracle={auc_o:.4f}"
          f"  Δ(oracle-baseline)={auc_o-auc_b:+.4f}  Δ(oracle-random)={auc_o-auc_r:+.4f}")

# ── Curvas Precision@theta ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
styles = {"Baseline": ("steelblue","--"), "Original (control)": ("gray","-."),
          "Random (lb)": ("tomato",":"), "Oracle (ub)": ("seagreen","-")}
for ax, nv in zip(axes, VIEW_GROUPS):
    for lbl, (df_, (color, ls)) in zip(
        ["Baseline","Original (control)","Random (lb)","Oracle (ub)"],
        zip([df_baseline, df_original, df_random, df_oracle], styles.values())
    ):
        errs = df_[df_["n_views"]==nv]["angular_error_deg"].fillna(90.0).tolist()
        thetas = np.linspace(0, 45, 300)
        precs  = [(np.array(errs) <= t).mean() for t in thetas]
        ax.plot(thetas, precs, label=lbl, color=color, linestyle=ls)
    ax.set_title(f"n_views={nv}"); ax.set_xlabel("Umbral (°)"); ax.set_ylabel("Fracción objetos")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("EXP-LIT-1: Candidates-then-Select — Precision@θ", fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig("lit1_candidates_precision_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: lit1_candidates_precision_curves.png")


---
## EXP-LIT-2 — Grid Overlay (superponer grilla de coordenadas)
*(MOKA arXiv:2403.03174; SketchVLM arXiv:2604.22875)*

**Motivación**: el sesgo X≈500 es un prior visual fuerte que el texto no puede
vencer (demostrado con v08). MOKA y SketchVLM muestran que superponer una grilla
de coordenadas visible sobre la imagen (igual a la notación de ajedrez) mejora el
grounding 8-15 % porque el modelo puede anclar su respuesta a una celda de la grilla
en lugar de estimar coordenadas libres — eliminando el prior de "el centro de la imagen".

**Implementación**: para cada render, se añade una grilla de N×N celdas con letras
en columnas (A, B, C...) y números en filas (1, 2, 3...). El prompt pide al modelo
que indique la celda de la grilla antes de dar la coordenada fina. Luego se convierte
la celda a coordenadas Molmo para la triangulación.

Como no podemos hacer nuevas inferencias aquí, simulamos el efecto midiendo cuánto
mejoraría el error si el modelo eligiera **la celda correcta** (el polo GT está en esa
celda) vs. si eligiera la celda más frecuente (la central, que simula el sesgo actual).
Esto da el mismo upper/lower bound que EXP-LIT-1, complementado con un análisis de
cuántas vistas actuales ya apuntan a la celda correcta (sin necesitar grilla).


In [ ]:
# ── EXP-LIT-2: Grid Overlay — análisis de celda y upper/lower bound ──────────
# Refs: MOKA arXiv:2403.03174; SketchVLM arXiv:2604.22875

GRID_N = 5    # grilla N×N (5x5 = 25 celdas, como las mayores usadas en MOKA)

# Letras para columnas (A-E para N=5)
GRID_COLS = [chr(65+i) for i in range(GRID_N)]
GRID_ROWS = list(range(1, GRID_N+1))


def molmo_to_cell(x: float, y: float, n: int = GRID_N) -> tuple[int, int]:
    """Convierte coordenada Molmo (0-1000) a índice de celda (col, row) en grilla NxN."""
    col = min(int(x / (1000 / n)), n-1)
    row = min(int(y / (1000 / n)), n-1)
    return col, row


def cell_to_molmo_center(col: int, row: int, n: int = GRID_N) -> tuple[float, float]:
    """Centro de una celda en coordenadas Molmo."""
    step = 1000 / n
    return (col + 0.5) * step, (row + 0.5) * step


def cell_name(col: int, row: int) -> str:
    return f"{GRID_COLS[col]}{GRID_ROWS[row]}"


def run_grid_experiment(strategy: str = "oracle") -> dict:
    """
    strategy:
        'oracle'   → usa la celda del GT proyectado (upper bound)
        'center'   → usa siempre la celda central C3 (simula sesgo X≈500 cuantizado)
        'original' → usa la celda del punto predicho por Molmo (control)
    """
    predictions = {}
    cell_stats = []  # para análisis de distribución de celdas

    for cat, oid in OBJECT_LIST_AXIS:
        gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.txt")
        gt_dir  = np.array(gt["elements"][0]["direction"])
        gt_orig = np.array(gt["elements"][0]["origin"])
        gt_point_top    = np.array(gt_orig) + 0.5 * gt_dir
        gt_point_bottom = np.array(gt_orig) - 0.5 * gt_dir

        render_dir = SANDBOX_RENDERS / "axis_sym" / oid / str(SIZE) / LIGHTING
        manifest_p = render_dir / "manifest.json"
        manifest   = json.load(open(manifest_p, encoding="utf-8")) if manifest_p.exists() else {}
        fov_deg    = manifest.get("fov", DEFAULT_FOV)
        image_size = manifest.get("image_size", SIZE)
        molmo_data = json.load(open(
            render_dir / f"molmo_multiview_{EXPERIMENT_ID_AXIS}.json", encoding="utf-8"))

        for n_views in VIEW_GROUPS:
            group      = molmo_data.get(str(n_views), {})
            pts_by_img = group.get("points_by_image", {})
            imgs_sent  = group.get("images_sent", [])

            center_col, center_row = GRID_N // 2, GRID_N // 2   # celda central (C3 para N=5)
            new_pts_by_img = {}

            for img_idx_str, pts in pts_by_img.items():
                pair = widest_pair(pts)
                if pair is None:
                    new_pts_by_img[img_idx_str] = []
                    continue
                p_a, p_b = pair
                cam = imgs_sent[int(img_idx_str)]
                R, T = cam["R"], cam["T"]

                # Proyección GT en esta vista
                gt_ax, gt_ay = project_3d_to_molmo(gt_point_top,    R, T, fov_deg, image_size)
                gt_bx, gt_by = project_3d_to_molmo(gt_point_bottom,  R, T, fov_deg, image_size)

                gt_col_a, gt_row_a = molmo_to_cell(gt_ax, gt_ay)
                gt_col_b, gt_row_b = molmo_to_cell(gt_bx, gt_by)
                pr_col_a, pr_row_a = molmo_to_cell(p_a["x"], p_a["y"])
                pr_col_b, pr_row_b = molmo_to_cell(p_b["x"], p_b["y"])

                hit_a = (gt_col_a == pr_col_a) and (gt_row_a == pr_row_a)
                hit_b = (gt_col_b == pr_col_b) and (gt_row_b == pr_row_b)
                cell_stats.append({
                    "object_id": oid, "n_views": n_views, "img_idx": int(img_idx_str),
                    "pred_cell_a": cell_name(pr_col_a, pr_row_a),
                    "gt_cell_a":   cell_name(gt_col_a, gt_row_a),
                    "hit_a": hit_a,
                    "pred_cell_b": cell_name(pr_col_b, pr_row_b),
                    "gt_cell_b":   cell_name(gt_col_b, gt_row_b),
                    "hit_b": hit_b,
                    "center_cell": cell_name(center_col, center_row),
                })

                if strategy == "oracle":
                    new_x_a, new_y_a = cell_to_molmo_center(gt_col_a, gt_row_a)
                    new_x_b, new_y_b = cell_to_molmo_center(gt_col_b, gt_row_b)
                elif strategy == "center":
                    new_x_a, new_y_a = cell_to_molmo_center(center_col, center_row)
                    new_x_b, new_y_b = cell_to_molmo_center(center_col, center_row)
                else:   # original
                    new_x_a, new_y_a = cell_to_molmo_center(pr_col_a, pr_row_a)
                    new_x_b, new_y_b = cell_to_molmo_center(pr_col_b, pr_row_b)

                new_pts_by_img[img_idx_str] = [
                    {"obj_id": 1, "x": new_x_a, "y": new_y_a},
                    {"obj_id": 2, "x": new_x_b, "y": new_y_b},
                ]

            try:
                pred = estimate_axis_no_mesh(new_pts_by_img, imgs_sent, fov_deg, image_size)
            except ValueError:
                pred = None
            predictions[(oid, n_views)] = pred

    return predictions, pd.DataFrame(cell_stats)


# ── Correr variantes ─────────────────────────────────────────────────────────
print("EXP-LIT-2: Corriendo variantes...")
preds_g_oracle,   df_cells_oracle   = run_grid_experiment("oracle")
preds_g_center,   df_cells_center   = run_grid_experiment("center")
preds_g_original, df_cells_original = run_grid_experiment("original")

df_g_oracle   = eval_axis_rows("EXP-LIT-2 Grid Oracle (ub)",       preds_g_oracle)
df_g_center   = eval_axis_rows("EXP-LIT-2 Grid Center (X≈500 sim)", preds_g_center)
df_g_original = eval_axis_rows("EXP-LIT-2 Grid Original (control)", preds_g_original)

s_g_oracle   = summarize_experiment("EXP-LIT-2 Grid Oracle (ub)",        df_g_oracle)
s_g_center   = summarize_experiment("EXP-LIT-2 Grid Center (sesgo sim)", df_g_center)
s_g_original = summarize_experiment("EXP-LIT-2 Grid Original (control)", df_g_original)

# ── Análisis de celdas: ¿cuántas vistas ya apuntan a la celda correcta? ────
print("\n=== Análisis de celdas (grilla 5×5) ===")
print(f"  Tasa de acierto celda polo A (pred == GT): "
      f"{df_cells_oracle['hit_a'].mean()*100:.1f}%")
print(f"  Tasa de acierto celda polo B (pred == GT): "
      f"{df_cells_oracle['hit_b'].mean()*100:.1f}%")
print(f"\n  Celdas más frecuentes polo A predicho:")
print(df_cells_oracle["pred_cell_a"].value_counts().head(8).to_string())
print(f"\n  Celdas GT polo A:")
print(df_cells_oracle["gt_cell_a"].value_counts().head(8).to_string())

# ── Comparación ──────────────────────────────────────────────────────────────
print("\n=== Comparación EXP-LIT-2 — AUC@45° ===")
comp_lit2 = pd.DataFrame({
    "Baseline":               {nv: round(auc_angular(df_baseline[df_baseline["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "Grid Center (X≈500)":    {nv: round(auc_angular(df_g_center[df_g_center["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "Grid Original (control)":{nv: round(auc_angular(df_g_original[df_g_original["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "Grid Oracle (ub)":       {nv: round(auc_angular(df_g_oracle[df_g_oracle["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
}).rename_axis("n_views")
print(comp_lit2.to_string())

# ── Heatmap de distribución de celdas predichas vs GT ───────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, col, title in [
    (axes[0], "pred_cell_a", "Celdas predichas polo A"),
    (axes[1], "gt_cell_a",   "Celdas GT polo A"),
]:
    heat = np.zeros((GRID_N, GRID_N))
    for cn, cnt in df_cells_oracle[col].value_counts().items():
        c, r = GRID_COLS.index(cn[0]), int(cn[1])-1
        heat[r, c] = cnt
    heat_pct = heat / heat.sum() * 100
    im = ax.imshow(heat_pct, cmap="YlOrRd")
    ax.set_xticks(range(GRID_N)); ax.set_xticklabels(GRID_COLS)
    ax.set_yticks(range(GRID_N)); ax.set_yticklabels(GRID_ROWS)
    for i in range(GRID_N):
        for j in range(GRID_N):
            ax.text(j, i, f"{heat_pct[i,j]:.0f}", ha="center", va="center", fontsize=8)
    ax.set_title(title); plt.colorbar(im, ax=ax, label="%")
plt.suptitle("EXP-LIT-2: Distribución de celdas en grilla 5×5", y=1.02)
plt.tight_layout()
plt.savefig("lit2_grid_cell_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: lit2_grid_cell_heatmap.png")


---
## EXP-LIT-3 — Structured CoT (razonamiento escena-grafo antes de señalar)
*(arXiv:2507.13362 — "Enhancing Spatial Reasoning in VLMs via CoT and RL")*

**Motivación**: el paper distingue CoT libre ("piensa paso a paso" — no ayuda o daña)
de CoT estructurado con scene-graph obligatorio (mejora significativamente el
razonamiento espacial). La estructura propuesta fuerza al modelo a declarar
explícitamente: tipo de objeto, orientación aproximada, y dirección esperada del eje
— antes de dar coordenadas.

**Implementación**: como no podemos hacer nuevas inferencias, simulamos el efecto
del CoT estructurado en dos formas:

1. **CoT perfecto**: asume que el modelo infirió correctamente el tipo de objeto
   y la orientación del eje (usando el GT), y usando esa orientación proyecta
   el eje al plano imagen para obtener la dirección correcta. Luego coloca los
   dos puntos en los extremos de esa proyección sobre el objeto. Upper bound.

2. **CoT aleatorio**: simula que el modelo eligió una orientación aleatoria del
   eje como prior. Lower bound / control.

3. **CoT por tipo de objeto**: agrupa los objetos del sandbox por forma aproximada
   (revolución, prismático, etc.) y usa la dirección modal de cada grupo como prior.
   Mide si un prior de grupo (que el CoT podría inferir correctamente del tipo)
   ayudaría sin necesitar GT individual.


In [ ]:
# ── EXP-LIT-3: Structured CoT — análisis con priors de orientación ───────────
# Ref: arXiv:2507.13362

COT_SEED = 123
rng_cot  = np.random.default_rng(COT_SEED)


def project_axis_to_image_segment(gt_dir: np.ndarray, gt_orig: np.ndarray,
                                    R: list, T: list, fov_deg: float,
                                    image_size: int, extent: float = 0.6) -> tuple | None:
    """
    Proyecta los dos extremos del eje GT (+/-extent desde el origen)
    al espacio Molmo. Devuelve ((x1,y1), (x2,y2)) o None si ambos fuera de imagen.
    """
    p1 = gt_orig + extent * gt_dir
    p2 = gt_orig - extent * gt_dir
    x1, y1 = project_3d_to_molmo(p1, R, T, fov_deg, image_size)
    x2, y2 = project_3d_to_molmo(p2, R, T, fov_deg, image_size)
    # Verificar que ambos estén en imagen
    in_image = lambda x, y: 5 <= x <= 995 and 5 <= y <= 995
    if not in_image(x1, y1) and not in_image(x2, y2):
        return None
    # Clip al borde
    x1 = float(np.clip(x1, 5, 995)); y1 = float(np.clip(y1, 5, 995))
    x2 = float(np.clip(x2, 5, 995)); y2 = float(np.clip(y2, 5, 995))
    return (x1, y1), (x2, y2)


def run_cot_experiment(strategy: str = "perfect") -> dict:
    """
    strategy:
        'perfect'   → usa la dirección GT como prior del CoT (upper bound)
        'random'    → usa una dirección 3D aleatoria como prior (lower bound)
        'modal'     → usa la dirección modal del grupo de objetos como prior
    """
    # Si modal: calcular la dirección media de los GT del grupo sandbox como proxy
    gt_dirs_all = []
    for cat, oid in OBJECT_LIST_AXIS:
        gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.txt")
        gt_dirs_all.append((oid, np.array(gt["elements"][0]["direction"])))

    # Dirección modal: promedio normalizado de todos los GT del sandbox
    modal_dir = np.mean([d for _, d in gt_dirs_all], axis=0)
    modal_dir /= np.linalg.norm(modal_dir)

    predictions = {}
    for cat, oid in OBJECT_LIST_AXIS:
        gt = parse_true_label(SANDBOX_OBJECTS / OBJECTS_SUBDIR_MAP["axis_sym"] / f"{oid}.txt")
        gt_dir  = np.array(gt["elements"][0]["direction"])
        gt_orig = np.array(gt["elements"][0]["origin"])

        if strategy == "perfect":
            prior_dir  = gt_dir
            prior_orig = gt_orig
        elif strategy == "random":
            # Dirección aleatoria uniformemente distribuida en la esfera
            v = rng_cot.standard_normal(3); v /= np.linalg.norm(v)
            prior_dir  = v
            prior_orig = np.zeros(3)
        else:  # modal
            prior_dir  = modal_dir
            prior_orig = np.zeros(3)

        render_dir = SANDBOX_RENDERS / "axis_sym" / oid / str(SIZE) / LIGHTING
        manifest_p = render_dir / "manifest.json"
        manifest   = json.load(open(manifest_p, encoding="utf-8")) if manifest_p.exists() else {}
        fov_deg    = manifest.get("fov", DEFAULT_FOV)
        image_size = manifest.get("image_size", SIZE)
        molmo_data = json.load(open(
            render_dir / f"molmo_multiview_{EXPERIMENT_ID_AXIS}.json", encoding="utf-8"))

        for n_views in VIEW_GROUPS:
            group      = molmo_data.get(str(n_views), {})
            pts_by_img = group.get("points_by_image", {})
            imgs_sent  = group.get("images_sent", [])

            new_pts_by_img = {}
            for img_idx_str, pts in pts_by_img.items():
                pair = widest_pair(pts)
                if pair is None:
                    new_pts_by_img[img_idx_str] = []
                    continue
                cam = imgs_sent[int(img_idx_str)]
                R, T = cam["R"], cam["T"]

                seg = project_axis_to_image_segment(prior_dir, prior_orig, R, T, fov_deg, image_size)
                if seg is None:
                    new_pts_by_img[img_idx_str] = []
                    continue
                (x1, y1), (x2, y2) = seg
                new_pts_by_img[img_idx_str] = [
                    {"obj_id": 1, "x": x1, "y": y1},
                    {"obj_id": 2, "x": x2, "y": y2},
                ]

            try:
                pred = estimate_axis_no_mesh(new_pts_by_img, imgs_sent, fov_deg, image_size)
            except ValueError:
                pred = None
            predictions[(oid, n_views)] = pred
    return predictions


# ── Correr variantes ─────────────────────────────────────────────────────────
print("EXP-LIT-3: Corriendo variantes...")
preds_cot_perfect = run_cot_experiment("perfect")
preds_cot_random  = run_cot_experiment("random")
preds_cot_modal   = run_cot_experiment("modal")

df_cot_perfect = eval_axis_rows("EXP-LIT-3 CoT Perfect (ub)", preds_cot_perfect)
df_cot_random  = eval_axis_rows("EXP-LIT-3 CoT Random (lb)",  preds_cot_random)
df_cot_modal   = eval_axis_rows("EXP-LIT-3 CoT Modal",        preds_cot_modal)

s_cot_perfect = summarize_experiment("EXP-LIT-3 CoT Perfect (ub)", df_cot_perfect)
s_cot_random  = summarize_experiment("EXP-LIT-3 CoT Random (lb)",  df_cot_random)
s_cot_modal   = summarize_experiment("EXP-LIT-3 CoT Modal",        df_cot_modal)

# ── Comparación ──────────────────────────────────────────────────────────────
print("\n=== Comparación EXP-LIT-3 — AUC@45° ===")
comp_lit3 = pd.DataFrame({
    "Baseline":          {nv: round(auc_angular(df_baseline[df_baseline["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "CoT Random (lb)":   {nv: round(auc_angular(df_cot_random[df_cot_random["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "CoT Modal":         {nv: round(auc_angular(df_cot_modal[df_cot_modal["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
    "CoT Perfect (ub)":  {nv: round(auc_angular(df_cot_perfect[df_cot_perfect["n_views"]==nv]["angular_error_deg"].tolist()), 4) for nv in VIEW_GROUPS},
}).rename_axis("n_views")
print(comp_lit3.to_string())


---
## Comparación global — los tres experimentos

In [ ]:
# ── Tabla global de upper bounds y lower bounds de los tres experimentos ─────
all_summaries = {
    "Baseline":                df_baseline,
    "LIT-1 Random (lb)":       df_random,
    "LIT-1 Oracle (ub)":       df_oracle,
    "LIT-2 Grid Center (X≈500)": df_g_center,
    "LIT-2 Grid Oracle (ub)":  df_g_oracle,
    "LIT-3 CoT Random (lb)":   df_cot_random,
    "LIT-3 CoT Modal":         df_cot_modal,
    "LIT-3 CoT Perfect (ub)":  df_cot_perfect,
}

print("=== AUC@45° global — todos los experimentos ===")
rows_global = []
for lbl, df_ in all_summaries.items():
    df_ = df_.copy(); df_["angular_error_deg"] = df_["angular_error_deg"].fillna(90.0)
    row = {"experimento": lbl}
    for nv in VIEW_GROUPS:
        errs = df_[df_["n_views"]==nv]["angular_error_deg"].tolist()
        row[f"AUC_nv{nv}"] = round(auc_angular(errs), 4)
        row[f"med_nv{nv}"] = round(float(np.median(errs)), 2)
    rows_global.append(row)

df_global = pd.DataFrame(rows_global).set_index("experimento")
display(df_global)

# Delta vs baseline
print("\n=== Delta AUC@45° vs Baseline ===")
for lbl in df_global.index:
    if lbl == "Baseline": continue
    deltas = [f"nv{nv}: {df_global.loc[lbl, f'AUC_nv{nv}'] - df_global.loc['Baseline', f'AUC_nv{nv}']:+.4f}"
              for nv in VIEW_GROUPS]
    tipo = "(ub)" if "Oracle" in lbl or "Perfect" in lbl else "(lb)" if "Random" in lbl else ""
    print(f"  {lbl:35s} {tipo:5s} {' | '.join(deltas)}")

# ── Curvas finales: mejor UB de cada experimento + baseline ─────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_series = [
    ("Baseline", df_baseline, "steelblue", "-"),
    ("LIT-1 Oracle (ub)",    df_oracle,      "seagreen",  "--"),
    ("LIT-2 Grid Oracle (ub)", df_g_oracle,  "darkorange", "--"),
    ("LIT-3 CoT Perfect (ub)", df_cot_perfect, "purple",  "--"),
    ("LIT-1 Random (lb)",    df_random,      "tomato",    ":"),
    ("LIT-3 CoT Modal",      df_cot_modal,   "gray",      "-."),
]
for ax, nv in zip(axes, VIEW_GROUPS):
    for lbl, df_, color, ls in plot_series:
        errs = df_[df_["n_views"]==nv]["angular_error_deg"].fillna(90.0).tolist()
        thetas = np.linspace(0, 45, 300)
        precs  = [(np.array(errs) <= t).mean() for t in thetas]
        ax.plot(thetas, precs, label=lbl, color=color, linestyle=ls)
    ax.set_title(f"n_views={nv}"); ax.set_xlabel("Umbral (°)"); ax.set_ylabel("Fracción objetos")
    ax.legend(fontsize=7); ax.grid(alpha=0.3)
plt.suptitle("Comparación global — Upper/Lower bounds de las tres propuestas (EXP-LIT-1/2/3)", y=1.02)
plt.tight_layout()
plt.savefig("lit_global_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Guardado: lit_global_comparison.png")

# ── Interpretación: ¿cuánto valdría implementar cada propuesta? ──────────────
print("\n=== Análisis de viabilidad — brecha UB vs Baseline ===")
for exp, ub_lbl in [("EXP-LIT-1", "LIT-1 Oracle (ub)"),
                     ("EXP-LIT-2", "LIT-2 Grid Oracle (ub)"),
                     ("EXP-LIT-3", "LIT-3 CoT Perfect (ub)")]:
    gains = [df_global.loc[ub_lbl, f"AUC_nv{nv}"] - df_global.loc["Baseline", f"AUC_nv{nv}"]
             for nv in VIEW_GROUPS]
    max_gain = max(gains)
    avg_gain = np.mean(gains)
    print(f"  {exp}: max Δ AUC={max_gain:+.4f}  avg Δ={avg_gain:+.4f}  "
          f"({'VALE LA PENA' if max_gain > 0.03 else 'MARGINAL'})")


---
## Interpretación de resultados

Las tres celdas anteriores producen los resultados de los experimentos.

**EXP-LIT-1 (Candidates-then-Select)**:
- El **Oracle (UB)** mide el máximo teórico de reformular pointing como selección:
  si la selección entre K candidatos fuera perfecta, el AUC mejoraría en Δ.
- El **Random (LB)** mide el piso: si la selección fuera al azar entre K candidatos
  generados aleatoriamente, el AUC bajaría en Δ respecto al baseline.
- La brecha Oracle-Random es la **señal de interés**: si es grande (>0.05 AUC),
  vale la pena implementar la selección real con Molmo2.

**EXP-LIT-2 (Grid Overlay)**:
- El heatmap de celdas muestra si el modelo ya acierta la celda correcta
  (tasa de acierto > 50 % → el sesgo X≈500 no domina completamente a nivel de celda).
- Si la tasa de acierto es baja (< 20 %), el overlay de grilla no resolvería
  el problema porque el modelo no inferiría la celda correcta de todas formas.
- El Oracle muestra el UB si la grilla funcionara perfectamente.

**EXP-LIT-3 (Structured CoT)**:
- El **CoT Perfect** muestra cuánto mejoraría si el CoT capturara perfectamente
  la orientación del eje antes de señalar.
- El **CoT Modal** simula usar el tipo de objeto como proxy (una silla siempre
  tiene eje vertical, un avión tiene eje horizontal, etc.).
- Si Modal >> Baseline, vale la pena diseñar un prompt CoT que infiera el tipo.

> Los resultados deben interpretarse como **análisis de viabilidad**, no como
> métricas reales: los UB requieren información GT que en producción no está
> disponible. La pregunta es si la brecha es suficientemente grande como para
> justificar el esfuerzo de implementación de cada propuesta.
